# LAB 07 — Advanced Machine Learning & Ensemble Methods

**Module:** Advanced Machine Learning & Ensemble Methods
**Session:** 07
**Duration:** 150 minutes (10-minute break included)
**Audience:** LAB02-LAB06'yı tamamlamış katılımcılar (başlangıç seviyesi ağırlıklı, 2 deneyimli katılımcı)

## LAB02-LAB06'dan Devam

- **LAB02-LAB03**: Pandas/NumPy, descriptive statistics, EDA.
- **LAB04**: `SimpleImputer`, `OneHotEncoder`, `StandardScaler`, `ColumnTransformer`, data leakage, Pipeline.
- **LAB05**: Regression metodolojisi (baseline → model → değerlendirme → karşılaştırma → seçim → tek seferlik test).
- **LAB06**: Classification metodolojisi — aynı E-Commerce Shipping veri kümesi, `DummyClassifier`,
  `LogisticRegression`, `DecisionTreeClassifier`, Confusion Matrix, Precision/Recall/F1, ROC-AUC,
  Average Precision, Class Imbalance, Decision Threshold Optimization.

Bu LAB'da önceki konuları baştan anlatmıyoruz — sadece kısa hatırlatmalarla kullanıyoruz.

## Bu LAB'ın Ana Hedefi

Bu LAB'ın amacı sadece Ensemble Learning algoritmalarını çalıştırmak değildir. Amaç, **sistematik
model karşılaştırması, deney tasarımı, genelleme performansı ve hesaplama maliyeti analizi**
öğretmektir.

## Bu LAB'ın Sonunda Yapabilecekleriniz

1. Ensemble Learning kavramını açıklayabileceksiniz.
2. Single Learner ve Ensemble Model arasındaki farkı anlayabileceksiniz.
3. Bootstrap Sampling mekanizmasını uygulayabileceksiniz.
4. Bagging ve Boosting arasındaki farkları açıklayabileceksiniz.
5. Random Forest modeli geliştirebileceksiniz.
6. AdaBoost çalışma mekanizmasını anlayabileceksiniz.
7. Gradient Boosting modeli geliştirebileceksiniz.
8. XGBoost ve LightGBM kullanabileceksiniz.
9. Bias–Variance Trade-off'u model davranışı üzerinden yorumlayabileceksiniz.
10. Temel hyperparameter'ların etkisini değerlendirebileceksiniz.
11. Farklı modelleri aynı deney koşullarında karşılaştırabileceksiniz.
12. Training Time ve Inference Latency ölçebileceksiniz.
13. Overfitting belirtilerini değerlendirebileceksiniz.
14. Model seçimini doğruluk, hız, karmaşıklık ve iş değeri açısından gerekçelendirebileceksiniz.

## Kullanılan Veri Kümesi

LAB06'da kullanılan **aynı** E-Commerce Shipping veri kümesi: `data/ecommerce_shipping/Train.csv`.
Yeni bir dataset indirilmeyecek, orijinal CSV değiştirilmeyecek.

**İş Problemi (LAB06 ile aynı): Shipment Delay Classification** — sevkiyatın zamanında ulaşıp
ulaşmayacağını tahmin eden farklı Classification modellerini karşılaştırmak.

**Önemli:** Bu LAB'ın amacı mümkün olan en yüksek accuracy'yi elde etmek değildir. Amaç, farklı
ensemble ailelerinin (Bagging, Boosting) nasıl çalıştığını, ne zaman yardımcı olduğunu (ve ne zaman
olmadığını) ve karşılaştırmalı benchmarking metodolojisini öğrenmektir.


---


## Section 1 — LAB06 Review & Ensemble Fundamentals
**Süre: 10 dakika**

### Learning Objective

Ensemble Learning kavramını, Single Learner'a kıyasla neden (ve ne zaman) işe yarayabileceğini
kavramsal olarak açıklayabilmek; Bagging ve Boosting arasındaki temel farkı anlayabilmek.

### Concept Explanation

**Single Learner**: LAB06'da tek bir `LogisticRegression` veya tek bir `DecisionTreeClassifier`
eğittik. Bu modellerin her biri **tek bir** hipotez (karar kuralı) üretir.

**Ensemble Learning**: Birden fazla modelin (base learner) tahminlerini birleştirerek tek bir
nihai tahmin üreten yaklaşımdır.

- **Weak Learner**: Tek başına rastgele tahminden biraz daha iyi performans gösteren basit bir
  model (örn. `max_depth=1` olan bir Decision Tree — "decision stump").
- **Strong Learner**: Tek başına yüksek performans gösteren bir model.
- **Model Diversity**: Ensemble'daki base learner'ların birbirinden farklı hatalar yapması.
  Ensemble'ın faydası büyük ölçüde bu çeşitlilikten gelir.
- **Bias**: Modelin, gerçek ilişkiyi sistematik olarak yanlış temsil etme eğilimi (örn. çok basit
  bir model, karmaşık bir ilişkiyi yakalayamaz — underfitting).
- **Variance**: Modelin, farklı training örneklemleri arasında ne kadar farklı sonuçlar üretme
  eğilimi (örn. çok karmaşık bir model, training verisindeki gürültüye bile aşırı duyarlı olur —
  overfitting).
- **Bagging (Bootstrap Aggregating)**: Base learner'lar **birbirinden bağımsız** olarak, farklı
  bootstrap örnekleri üzerinde eğitilir; tahminler ortalanır (regression) veya oylanır
  (classification).
- **Boosting**: Base learner'lar **sırayla**, her biri öncekinin hatalarını düzeltmeye çalışacak
  şekilde eğitilir.
- **Voting / Stacking (sadece kavramsal tanıtım — bu LAB'da uygulanmayacak, Challenge 4'te kısaca
  ele alınacak)**: Voting, farklı türde modellerin tahminlerini (oylama veya olasılık ortalaması
  ile) birleştirir; Stacking, base model tahminlerini bir "meta-model"e girdi olarak verir. Bu
  LAB'ın ana uygulaması Bagging ve Boosting'e odaklanır.

### Important Technical Distinctions

- **Ensemble Learning her zaman performansı artırmaz.**
- **Daha fazla estimator kullanmak her zaman daha iyi sonuç vermez** — belirli bir noktadan sonra
  marjinal katkı küçülür (Section 3/Challenge 1'de bunu sayılarla göreceğiz).
- **Model diversity önemlidir** — birbirine çok benzeyen hatalar üreten modelleri birleştirmek
  sınırlı fayda sağlar.
- **Bagging genellikle variance azaltmaya yardımcı olur** (bağımsız, çeşitlendirilmiş learner'ların
  ortalaması, tek bir karmaşık modelin gürültüye duyarlılığını yumuşatır).
- **Boosting çoğu durumda bias azaltabilir**; ancak variance ve overfitting üzerindeki etkisi
  koşullara (learning rate, estimator sayısı, veri gürültüsü) bağlıdır — kesin bir kural değildir.

| Bagging | Boosting |
|---|---|
| Base learner'lar bağımsız eğitilebilir | Base learner'lar aşamalı biçimde geliştirilir |
| Çeşitlilik ve aggregation | Önceki modelin eksiklerini giderme |
| Variance azaltmaya yardımcı olabilir | Bias azaltmaya yardımcı olabilir |
| Random Forest örneği | AdaBoost ve Gradient Boosting örnekleri |

Bu tablo, kesin matematiksel bir kural değil, genel bir öğretim çerçevesidir.

### Educational Question

**Tek bir Decision Tree yüksek training accuracy elde ediyor ancak validation performansı düşük
kalıyorsa, Ensemble Learning nasıl yardımcı olabilir?**

Bu soruyu şimdi tartışın (kesin cevabı yok — fikrinizi not edin). Section 2'de, bu LAB'ın gerçek
veri kümesinde **tam olarak bu senaryoyu** (derin bir Decision Tree'nin training/validation
accuracy farkı) sayılarla göreceğiz ve Section 2-3'te Bagging/Random Forest'ın bu duruma nasıl
yaklaştığını inceleyeceğiz.


---


## Section 2 — Baseline Models, Bootstrap & Bagging
**Süre: 15 dakika**

### Learning Objective

LAB06'daki dataset/split/preprocessing yaklaşımını bu LAB için yeniden kurmak; baseline modelleri
yeniden hesaplamak; Bootstrap Sampling mekanizmasını somut bir örnekle görmek; Bagging'in tek bir
Decision Tree'ye kıyasla etkisini gözlemlemek.

### Part A — Dataset & Split Preparation

LAB06'da kullanılan feature ve target tanımlarını, aynı split stratejisiyle (stratified,
`random_state=42`, `%70/%15/%15`) yeniden kuruyoruz.


In [ ]:
# Same environment setup pattern as LAB02-LAB06: resolve the project root, add src/ to sys.path,
# then load the shared configuration and the dataset.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


In [ ]:
from data_loading import load_config, load_ecommerce_shipping

config = load_config()
RANDOM_STATE = config["lab07"]["random_seed"]
TARGET = config["lab07"]["target"]
POSITIVE_CLASS = config["lab07"]["positive_class"]
DECISION_THRESHOLD = config["lab07"]["decision_threshold"]
BENCHMARK_THREADS = config["lab07"]["benchmark"]["threads"]

df = load_ecommerce_shipping(config)
print("Dataset shape:", df.shape)
print("Target column:", TARGET, "| Positive class:", POSITIVE_CLASS)
print("Random state (from config/settings.yaml):", RANDOM_STATE)
print("Common decision threshold for the main benchmark:", DECISION_THRESHOLD)
print("Fixed CPU thread policy (n_jobs for supporting estimators):", BENCHMARK_THREADS)


LAB06'daki **aynı** feature seçim kararlarını kullanıyoruz (gerekçeler orada belgelenmiştir):
`ID`, target, `Customer_rating`, `Customer_care_calls` (tahmin anındaki erişilebilirliği
doğrulanmamış / leakage riski) ve `Gender` (fairness riski, operasyonel gerekçe yok) varsayılan
model girdilerine dahil edilmez.


In [ ]:
numerical_features = ["Cost_of_the_Product", "Prior_purchases", "Discount_offered", "Weight_in_gms"]
categorical_features = ["Warehouse_block", "Mode_of_Shipment", "Product_importance"]
feature_columns = numerical_features + categorical_features

X = df[feature_columns]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature columns:", feature_columns)


In [ ]:
import matplotlib.pyplot as plt

# Color palette: fixed categorical/semantic roles reused across the ml-labs notebook series.
BAR_COLOR = "#2a78d6"
HIGHLIGHT_COLOR = "#1baf7a"
WARNING_COLOR = "#eb6834"
GRID_COLOR = "#e1e0d9"


def style_axes(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)


### Data Splitting & Test Set Integrity

Aynı random seed değeri, iki ayrı çalıştırmanın **mutlaka aynı** veri bölmelerini ürettiği anlamına
gelmez — bu, split fonksiyonuna verilen input DataFrame'in sırası, index'i ve parametreleri de
aynıysa geçerlidir. Burada LAB06'nın split kodunu **birebir aynı** şekilde (aynı feature/target
tanımı, aynı `test_size` değerleri, aynı `stratify`, aynı `random_state=42`) çalıştırıyoruz; bu
yüzden üretilecek üç kümenin gerçekten LAB06'dakiyle aynı olup olmadığını index seviyesinde
**doğrulayacağız** (varsaymayacağız).

### Hands-On


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} | n={len(yy):5d} | positive rate: {yy.mean():.4f}")


In [ ]:
# Index-level check: is this split identical to LAB06's split, or merely similar?
print("Train index sample (sorted, first 5):", sorted(X_train.index.tolist())[:5])
print("Val   index sample (sorted, first 5):", sorted(X_val.index.tolist())[:5])
print("Test  index sample (sorted, first 5):", sorted(X_test.index.tolist())[:5])
print()
print("Index overlap train/val :", len(set(X_train.index) & set(X_val.index)))
print("Index overlap train/test:", len(set(X_train.index) & set(X_test.index)))
print("Index overlap val/test  :", len(set(X_val.index) & set(X_test.index)))


### CRITICAL — Test Set Integrity Notice

LAB06, Section 8'de **bu dataset'in `Reached.on.Time_Y.N` hedefi için, birebir aynı split
mekanizmasıyla** (`test_size=0.30`/`0.50`, `stratify`, `random_state=42`) üretilen test kümesini
**zaten bir kez** kullanmış ve raporlamıştır (`LogisticRegression` + seçilmiş threshold ile).
Buradaki `X_train`/`X_val`/`X_test` tam olarak LAB06'dakiyle **aynı satırlardan** oluşur (shape ve
positive rate değerleri LAB06'nın raporladığı sayılarla eşleşiyor: train ≈ %59.67, val ≈ %59.70).

**Sonuç:** Bu LAB'daki `X_test`/`y_test`, gerçekten görülmemiş, bağımsız bir holdout **değildir** —
LAB06'da zaten bir model + threshold kararına dayanak olmuştur. Bu, aynı kayıtların yeni bir
random split ile tekrar bölünmesinin önceki test kullanımını ortadan kaldırmadığının somut bir
örneğidir.

Bu nedenle bu LAB'da:

1. **Bütün model seçim/ranking kararları yalnızca validation verisi (`X_val`/`y_val`) üzerinden**
   verilecektir (Section 6-7).
2. Section 7'de test verisi üzerinde yapılacak herhangi bir inceleme, **retrospective** (geriye
   dönük, karar değiştirmeyen) olarak açıkça etiketlenecektir — "nihai, bağımsız bir test
   evaluation" olarak sunulmayacaktır.
3. Test verisi, bu LAB'da **hiçbir model veya hyperparameter seçiminde** kullanılmayacaktır.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: y_train, y_val, y_test pozitif sınıf oranlarının (yukarıdaki çıktı) orijinal dağılıma
#       (df[TARGET].mean()) ne kadar yakın olduğunu karşılaştırın.
# TODO: Index overlap değerlerinin neden hepsinin 0 olması gerektiğini açıklayın.

# print("Original positive rate:", df[TARGET].mean())


### Part B — Baseline Models

LAB06'daki baseline modelleri (DummyClassifier, LogisticRegression, DecisionTreeClassifier) bu
LAB'ın kendi split'i üzerinde **yeniden hesaplıyoruz** (önceki LAB'dan hardcode edilmiş sonuç
kullanılmıyor).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_model = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

logreg_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=config["lab07"]["models"]["logistic_regression"]["max_iter"], random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

tree_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=config["lab07"]["models"]["decision_tree"]["max_depth"], random_state=RANDOM_STATE)),
]).fit(X_train, y_train)


def quick_report(name, pred_val):
    print(f"{name:30s} | Val Acc={accuracy_score(y_val, pred_val):.4f}  "
          f"Precision={precision_score(y_val, pred_val, zero_division=0):.4f}  "
          f"Recall={recall_score(y_val, pred_val, zero_division=0):.4f}  "
          f"F1={f1_score(y_val, pred_val, zero_division=0):.4f}")


quick_report("DummyClassifier(most_frequent)", baseline_model.predict(X_val))
quick_report("LogisticRegression", logreg_model.predict(X_val))
quick_report("DecisionTree(max_depth=5)", tree_model.predict(X_val))


### Hands-On — Single Decision Tree: Overfitting Illustration

Section 1'deki Educational Question'ı şimdi gerçek verilerle cevaplıyoruz: `max_depth`
sınırlanmamış (yani ağacın büyümesi kısıtlanmamış) bir Decision Tree ile sınırlı derinlikli bir
Decision Tree'nin training/validation accuracy farkını karşılaştıralım.


In [ ]:
deep_tree = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

shallow_tree = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

for name, pipe in [("Deep tree (max_depth=None)", deep_tree), ("Shallow tree (max_depth=3)", shallow_tree)]:
    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    val_acc = accuracy_score(y_val, pipe.predict(X_val))
    print(f"{name:28s} | Train Acc={train_acc:.4f}  Val Acc={val_acc:.4f}  Gap={train_acc - val_acc:.4f}")


### Interpretation

`max_depth=None` olan ağaç, training verisini **ezberliyor** (Train Acc = 1.0000) ama validation
performansı (≈0.6412) çok daha düşük — **%35.9 puanlık bir generalization gap**, klasik bir
overfitting örneği. Sınırlı derinlikli (`max_depth=3`) ağaç ise çok daha küçük bir gap (≈%1.3)
gösteriyor, çünkü modelin karmaşıklığı veri miktarına göre daha dengeli.

Bu, Section 1'in sorusuna somut bir zemin hazırlıyor: **Derin, aşırı uydurulmuş bir tek ağacı
doğrudan kullanmak yerine**, Section 2 Part D ve Section 3'te göreceğimiz gibi, birden fazla
(genellikle sınırlı derinlikli) ağacı bootstrap örnekleriyle çeşitlendirip birleştirmek
(Bagging/Random Forest), variance'ı azaltarak daha dengeli bir validation performansı elde etmeye
çalışır.

### Part C — Bootstrap Sampling

Bagging'in temelini oluşturan **sampling with replacement** mekanizmasını küçük bir NumPy array
üzerinde gösterelim.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

population = np.arange(1, 11)

bootstrap_sample = rng.choice(
    population,
    size=len(population),
    replace=True,
)

print("Original population:", population)
print("Bootstrap sample    :", bootstrap_sample)
print()
print("Unique values selected:", sorted(set(bootstrap_sample.tolist())))
print("Count of unique values selected:", len(set(bootstrap_sample.tolist())), "out of", len(population))
print("Values never selected in this sample:", sorted(set(population.tolist()) - set(bootstrap_sample.tolist())))


### Interpretation

Bu tek bootstrap örneğinde (`random_state=42`), 10 elemanlı popülasyondan yine 10 elemanlık bir
örnek çekildi, ancak:

- **Bazı değerler birden fazla seçildi** (örn. `1` üç kez, `5` ve `7` ikişer kez seçildi).
- **Bazı değerler hiç seçilmedi** (`2`, `4`, `6`, `10` — bu örnekte 4 değer).
- **Bootstrap sample size** her zaman orijinal popülasyon büyüklüğüyle aynıdır (`size=len(population)`).
- Her yeni bootstrap çekilişinde (farklı bir rastgele durumla), **farklı** bir alt küme ve farklı
  tekrar sayıları elde edilir — bu, **sample diversity**'nin kaynağıdır: her base learner, biraz
  farklı bir veri görünümü üzerinde eğitilir.

### Part D — Bagging Demonstration


In [ ]:
from sklearn.ensemble import BaggingClassifier

bagging_cfg = config["lab07"]["models"]["bagging"]

bagging_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=bagging_cfg["base_max_depth"]),
        n_estimators=bagging_cfg["n_estimators"],
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=BENCHMARK_THREADS,
    )),
]).fit(X_train, y_train)

print("BaggingClassifier fitted successfully.")
print("scikit-learn version (API compatibility check):", __import__("sklearn").__version__)


In [ ]:
from sklearn.metrics import roc_auc_score

for name, pipe in [("Single tree (max_depth=3)", shallow_tree), ("Bagging (20x depth=3 trees)", bagging_model)]:
    pred_val = pipe.predict(X_val)
    proba_val = pipe.predict_proba(X_val)[:, 1]
    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    val_acc = accuracy_score(y_val, pred_val)
    print(f"{name:30s} | Train Acc={train_acc:.4f}  Val Acc={val_acc:.4f}  "
          f"Val F1={f1_score(y_val, pred_val):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val):.4f}")


### Interpretation

20 adet `max_depth=3` ağacından oluşan Bagging ensemble'ı, tek bir `max_depth=3` ağacına kıyasla
**küçük ama gerçek bir iyileşme** gösteriyor (Val Acc, Val F1 ve Val ROC-AUC'nin hepsinde). Bu,
Section 1'de vurgulanan önemli bir noktayı doğruluyor: **Bagging'in faydası burada dramatik değil**
— çünkü base learner'lar zaten düşük derinlikte ve nispeten kararlı; Bagging'in asıl gücünü
Section 3'te, daha fazla ağaç ve random feature selection ile (Random Forest) göreceğiz.

Bu `BaggingClassifier` demonstrasyonu eğitim amaçlıdır; Section 6'nın ana benchmark tablosuna
dahil edilmesi zorunlu değildir (farkı zaten burada gösterdik).

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: bagging_cfg["n_estimators"] değerini (örn. 5 veya 50) değiştirip BaggingClassifier'ı
#       yeniden eğitin.
# TODO: Val Acc / Val F1 / Val ROC-AUC değerlerinin estimator sayısıyla nasıl değiştiğini gözlemleyin.

# bagging_small = Pipeline(steps=[("preprocessing", preprocessor), ("model", BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=5, bootstrap=True, random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS))]).fit(X_train, y_train)
# print(accuracy_score(y_val, bagging_small.predict(X_val)))


---


## Section 3 — Random Forest
**Süre: 20 dakika**

### Learning Objective

Random Forest'ın, klasik Bagging'e kıyasla ek bir çeşitlendirme mekanizması (random feature
selection) kullandığını anlamak; `n_estimators` ve `max_depth` hyperparametrelerinin model
davranışı üzerindeki etkisini gerçek sayılarla gözlemlemek; Out-of-Bag (OOB) değerlendirmesinin ne
olduğunu ve sınırlarını bilmek.

### Concept Explanation

Section 2'deki `BaggingClassifier`, her ağacı farklı bir **bootstrap örneği** üzerinde eğiterek
çeşitlendirme sağladı. **Random Forest**, buna ek bir çeşitlendirme katmanı ekler: her **split**
aşamasında, ağacın bakabileceği feature'lar da **rastgele bir alt kümeyle** (`max_features`)
sınırlanır. Bu, ağaçlar arasındaki **korelasyonu düşürür** — eğer tüm ağaçlar her split'te en güçlü
feature'ı (örn. `Discount_offered`) görebilseydi, çoğu ağaç çok benzer kararlar verir ve
**correlated errors** üretirdi; bu da ensemble ortalamasının variance azaltma faydasını zayıflatır.

- **Bootstrap Sampling**: Section 2 Part C'deki aynı mekanizma — her ağaç farklı bir satır alt
  kümesi görür.
- **Random Feature Selection**: Her split'te, tüm feature'lar değil, rastgele seçilmiş bir alt
  küme (`max_features`) değerlendirilir.
- **Multiple Decision Trees / Prediction Aggregation**: `RandomForestClassifier`, `predict_proba()`
  için ağaçların olasılık tahminlerinin **ortalamasını** alır (Scikit-learn implementasyonu); hard
  `predict()` ise bu ortalanmış olasılığa göre sınıfı belirler.
- **Ensemble Diversity / Correlated Errors**: Yukarıda açıklandı.
- **Out-of-Bag (OOB) Samples / OOB Score**: Her ağaç, bootstrap örneklemesi sırasında seçilmeyen
  (`~%37` ortalamada) satırları (**OOB samples**) hiç görmez — bu satırlar, o ağaç için "ücretsiz"
  bir validation kümesi gibi kullanılabilir. `oob_score=True` ile, her satırın tahmini **yalnızca
  o satırı OOB olarak bırakan ağaçlardan** ortalanarak hesaplanan bir OOB skoru elde edilir.
- **Feature Importance**: Her feature'ın, ormandaki ağaçlarda ortalama olarak ne kadar impurity
  azaltımına katkı sağladığının bir özeti (Mean Decrease in Impurity).
- **Model Complexity**: `n_estimators`, `max_depth`, `min_samples_leaf` gibi parametrelerle
  kontrol edilir.

### Hands-On


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_cfg = config["lab07"]["models"]["random_forest"]
print("Random Forest starting configuration (educational starting point, NOT claimed optimal):", rf_cfg)

random_forest_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=rf_cfg["n_estimators"],
        max_depth=rf_cfg["max_depth"],
        min_samples_leaf=rf_cfg["min_samples_leaf"],
        max_features=rf_cfg["max_features"],
        random_state=RANDOM_STATE,
        n_jobs=BENCHMARK_THREADS,
    )),
]).fit(X_train, y_train)

pred_val_rf = random_forest_model.predict(X_val)
proba_val_rf = random_forest_model.predict_proba(X_val)[:, 1]
train_acc_rf = accuracy_score(y_train, random_forest_model.predict(X_train))
print(f"Train Acc={train_acc_rf:.4f}  Val Acc={accuracy_score(y_val, pred_val_rf):.4f}  "
      f"Val F1={f1_score(y_val, pred_val_rf):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val_rf):.4f}")


Bu konfigürasyon **optimal değerler olarak sunulmamaktadır** — eğitim amaçlı, mantıklı bir
başlangıç konfigürasyonudur. Section 3-5'teki deneyler ve Section 6'nın benchmark'ı, farklı
konfigürasyonların gerçek etkisini ölçmeyi amaçlar.

### Hyperparameter Explanation

- **`n_estimators`**: Ormandaki ağaç sayısı.
- **`max_depth`**: Her ağacın maksimum derinliği (karmaşıklık sınırı).
- **`min_samples_split`**: Bir düğümün bölünebilmesi için gereken minimum örnek sayısı
  (bu LAB'da varsayılan değer kullanılıyor).
- **`min_samples_leaf`**: Bir yaprak düğümde olması gereken minimum örnek sayısı.
- **`max_features`**: Her split'te değerlendirilecek rastgele feature alt kümesinin büyüklüğü
  (`"sqrt"` → toplam feature sayısının karekökü).
- **`bootstrap`**: Bootstrap örneklemesinin kullanılıp kullanılmayacağı (varsayılan `True`).
- **`random_state`**: Tekrar üretilebilirlik.
- **`n_jobs`**: Paralel çalıştırılacak iş parçacığı sayısı (bu LAB'da `config/settings.yaml`'daki
  `benchmark.threads` ile sabitlenmiştir — kontrolsüz `n_jobs=-1` kullanılmamaktadır).

### Experiment 1 — Number of Trees


In [ ]:
from time import perf_counter

n_estimators_values = [10, 50, 100, 200]
exp1_rows = []

for n in n_estimators_values:
    start = perf_counter()
    model = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=n,
            max_depth=rf_cfg["max_depth"],
            min_samples_leaf=rf_cfg["min_samples_leaf"],
            max_features=rf_cfg["max_features"],
            random_state=RANDOM_STATE,
            n_jobs=BENCHMARK_THREADS,
        )),
    ])
    model.fit(X_train, y_train)
    training_time = perf_counter() - start

    pred_tr = model.predict(X_train)
    pred_v = model.predict(X_val)
    proba_v = model.predict_proba(X_val)[:, 1]
    exp1_rows.append({
        "n_estimators": n,
        "Train_Accuracy": accuracy_score(y_train, pred_tr),
        "Validation_Accuracy": accuracy_score(y_val, pred_v),
        "Validation_F1": f1_score(y_val, pred_v),
        "Validation_ROC_AUC": roc_auc_score(y_val, proba_v),
        "Training_Time_s": training_time,
    })

exp1_df = pd.DataFrame(exp1_rows).round(4)
exp1_df


### Experiment 2 — Tree Depth


In [ ]:
max_depth_values = [3, 8, None]
exp2_rows = []

for d in max_depth_values:
    model = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=rf_cfg["n_estimators"],
            max_depth=d,
            min_samples_leaf=rf_cfg["min_samples_leaf"],
            max_features=rf_cfg["max_features"],
            random_state=RANDOM_STATE,
            n_jobs=BENCHMARK_THREADS,
        )),
    ])
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    exp2_rows.append({
        "max_depth": str(d),
        "Train_Accuracy": train_acc,
        "Validation_Accuracy": val_acc,
        "Generalization_Gap": train_acc - val_acc,
    })

exp2_df = pd.DataFrame(exp2_rows).round(4)
exp2_df


### Visualization


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(exp1_df["n_estimators"], exp1_df["Validation_Accuracy"], marker="o", color=BAR_COLOR, linewidth=2, label="Validation Accuracy")
ax.plot(exp1_df["n_estimators"], exp1_df["Validation_F1"], marker="o", color=HIGHLIGHT_COLOR, linewidth=2, label="Validation F1")
ax.plot(exp1_df["n_estimators"], exp1_df["Validation_ROC_AUC"], marker="o", color=WARNING_COLOR, linewidth=2, label="Validation ROC-AUC")
ax.set_title("Number of Trees vs Validation Performance")
ax.set_xlabel("n_estimators")
ax.set_ylabel("Score")
ax.legend(frameon=False)
style_axes(ax)
plt.show()


In [ ]:
depth_labels = exp2_df["max_depth"]
x_pos = np.arange(len(depth_labels))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_pos, exp2_df["Train_Accuracy"], marker="o", color=BAR_COLOR, linewidth=2, label="Train Accuracy")
ax.plot(x_pos, exp2_df["Validation_Accuracy"], marker="o", color=WARNING_COLOR, linewidth=2, label="Validation Accuracy")
ax.set_xticks(x_pos)
ax.set_xticklabels(depth_labels)
ax.set_title("Tree Depth vs Training/Validation Performance")
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.legend(frameon=False)
style_axes(ax)
plt.show()


### Interpretation

**Experiment 1 (Number of Trees):** Validation Accuracy/F1/ROC-AUC, `n_estimators=10`'dan
`200`'e çıkarken **monoton biçimde iyileşmiyor** — `n_estimators=100` en yüksek Val Accuracy'yi
(0.6715) verirken, `n_estimators=200`'de Val Accuracy hafifçe düşüyor (0.6685) ve Val F1 aslında
`n_estimators=10`'da (0.6766) en yüksek. Bu, **"daha fazla ağaç her zaman daha iyi sonuç verir"**
iddiasının bu dataset için **geçerli olmadığını** gösteriyor — farklar küçük ve büyük ölçüde
gürültü düzeyinde. Training Time ise `n_estimators` ile **yaklaşık doğrusal** artıyor (10 ağaçta
≈0.04s'den 200 ağaçta ≈0.36s'ye). Yani bu deneyde ek ağaçlar, anlamlı bir performans kazancı
getirmeden hesaplama maliyetini artırıyor — Challenge 1'de bu "diminishing returns" ilişkisini
daha sistematik inceleyeceğiz.

**Experiment 2 (Tree Depth):** Burada daha belirgin bir örüntü var — `max_depth=3`
(**Generalization Gap = -0.0135**, yani validation accuracy training'den bile hafifçe yüksek,
muhtemelen küçük örnekleme gürültüsü) en iyi Validation Accuracy'yi (0.6879) veriyor;
`max_depth=8` (Gap=0.0703) ve özellikle `max_depth=None` (Train Acc=0.9812, Gap=0.3345) ile
**overfitting belirtisi güçleniyor**. **Önemli:** Bu, "ağaç sayısı arttıkça model mutlaka daha
iyi/overfit olur" gibi genel bir kural değildir — bu spesifik dataset ve feature seti için
gözlemlenen bir sonuçtur; farklı bir veri kümesinde farklı bir örüntü gözlemlenebilir.

### Out-of-Bag Discussion


In [ ]:
rf_oob_model = RandomForestClassifier(
    n_estimators=rf_cfg["n_estimators"],
    max_depth=rf_cfg["max_depth"],
    min_samples_leaf=rf_cfg["min_samples_leaf"],
    max_features=rf_cfg["max_features"],
    random_state=RANDOM_STATE,
    n_jobs=BENCHMARK_THREADS,
    oob_score=True,
)

# NOTE: oob_score requires fitting the estimator directly (not through a Pipeline wrapper that
# re-fits the preprocessor separately), so we fit the preprocessor once and reuse its output here.
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_val_transformed = preprocessor.transform(X_val)
rf_oob_model.fit(X_train_transformed, y_train)

independent_val_acc = accuracy_score(y_val, rf_oob_model.predict(X_val_transformed))
print(f"OOB score (from training data only)     : {rf_oob_model.oob_score_:.4f}")
print(f"Accuracy on independent validation set   : {independent_val_acc:.4f}")


OOB skoru (0.6785) ile bağımsız validation accuracy'si (0.6715) **birbirine yakın** — bu, OOB
değerlendirmesinin burada validation performansı için makul bir yaklaşık (proxy) sağladığını
gösteriyor. **Ancak bu iki değer özdeş değildir ve OOB skoru bağımsız bir test performansının
yerine geçmez** — özellikle `preprocessor`, yukarıdaki hücrede **tüm training verisine** fit
edildiği için (OOB mantığının gerektirdiği gibi ayrı ayrı her ağaç için değil), OOB değerlendirmesi
preprocessing açısından tamamen bağımsız değildir. Pratikte OOB, hızlı bir sağlık kontrolü olarak
kullanışlıdır, ama bu LAB'daki asıl model seçim kararları **validation seti üzerinden** verilecektir.

### Hands-On — Feature Importance


In [ ]:
feature_names_rf = random_forest_model.named_steps["preprocessing"].get_feature_names_out()
feature_importances = pd.Series(
    random_forest_model.named_steps["model"].feature_importances_, index=feature_names_rf
).sort_values(ascending=False)

print("Top 5 features by Random Forest Mean Decrease in Impurity:")
print(feature_importances.head(5))


`Discount_offered` (≈%45.6) ve `Weight_in_gms` (≈%34.8) toplamda ormanın impurity azaltımının
**%80'inden fazlasını** açıklıyor — bu, LAB06'daki Logistic Regression katsayı analiziyle
(en büyük katsayılar aynı iki feature'daydı) **yönsel olarak tutarlı**, farklı bir model ailesinden
bağımsız bir doğrulama.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: rf_cfg["max_features"] değerini "sqrt" yerine None (tüm feature'lar her split'te
#       değerlendirilir) olarak değiştirip RandomForestClassifier'ı yeniden eğitin.
# TODO: Val Accuracy/F1/ROC-AUC değerlerinin nasıl değiştiğini ve bunun "ensemble diversity"
#       kavramıyla ilişkisini yorumlayın.

# rf_all_features = Pipeline(steps=[("preprocessing", preprocessor), ("model", RandomForestClassifier(n_estimators=rf_cfg["n_estimators"], max_depth=rf_cfg["max_depth"], min_samples_leaf=rf_cfg["min_samples_leaf"], max_features=None, random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS))]).fit(X_train, y_train)
# print(accuracy_score(y_val, rf_all_features.predict(X_val)))


---


## Section 4 — AdaBoost & Gradient Boosting
**Süre: 30 dakika**

### Part A — Boosting Fundamentals

### Learning Objective

Boosting'in Bagging'den temel farkını (sıralı öğrenme, hata düzeltme) anlamak.

### Concept Explanation

- **Sequential Learning**: Boosting'teki her base learner, kendisinden önceki learner'ların
  ürettiği hatalara bakılarak eğitilir — Bagging'deki gibi bağımsız değildir.
- **Weak Learners**: Boosting genellikle çok basit base learner'lar (örn. `max_depth=1` "decision
  stump") kullanır; gücünü tekil modellerden değil, **art arda biriken düzeltmelerden** alır.
- **Error Correction**: Her adım, önceki adımın hatalı/zor örneklerine daha fazla odaklanır.
- **Weighted Contributions / Additive Modeling**: Nihai tahmin, her adımın katkısının **ağırlıklı
  toplamıdır** (AdaBoost'ta örnek ağırlıkları ve estimator ağırlıkları ile; Gradient Boosting'te
  learning rate ile ölçeklenen artımlı modeller ile).
- **Learning Rate**: Her adımın nihai modele ne kadar katkıda bulunacağını ölçekler (küçük
  learning rate = daha fazla adım gerekir, ama genellikle daha kararlı bir öğrenme).
- **Number of Estimators**: Sıralı adım sayısı.

| Bagging | Boosting |
|---|---|
| Base learner'lar bağımsız eğitilebilir | Base learner'lar aşamalı biçimde geliştirilir |
| Çeşitlilik ve aggregation | Önceki modelin eksiklerini giderme |
| Variance azaltmaya yardımcı olabilir | Bias azaltmaya yardımcı olabilir |
| Random Forest örneği | AdaBoost ve Gradient Boosting örnekleri |

Bu, kesin bir matematiksel kanıt değil, **genel bir öğretim çerçevesidir** — Section 6'da bu iki
ailenin bu spesifik dataset üzerindeki gerçek davranışını ölçeceğiz.

### Part B — AdaBoost

### Concept Explanation

- **Adaptive Boosting (AdaBoost)**: Her iterasyonda, **önceki modelin yanlış sınıflandırdığı**
  gözlemlerin **sample weight**'i artırılır — böylece sonraki base learner, bu "zor" örneklere
  daha fazla önem vermek zorunda kalır.
- **Decision Stump**: AdaBoost'ta yaygın olarak kullanılan, `max_depth=1` bir Decision Tree —
  klasik bir weak learner.
- **Weighted Voting**: Her base learner'ın nihai oya katkısı, kendi performansına bağlı bir
  ağırlıkla (`alpha`) ölçeklenir — daha başarılı learner'lar daha fazla söz hakkına sahiptir.
- **Sensitivity to Noise**: Yanlış etiketlenmiş (gürültülü) gözlemler, art arda artan ağırlıklar
  nedeniyle AdaBoost'u orantısız biçimde etkileyebilir.

### Hands-On — Sample Weight Update (Synthetic Example)

**[SYNTHETIC] Aşağıdaki örnek, gerçek E-Commerce Shipping verisinden değildir** — AdaBoost'un
ağırlık güncelleme mekanizmasını küçük, kontrollü bir örnekle göstermek içindir.


In [ ]:
# [SYNTHETIC] Tiny 1-D toy example — illustrates AdaBoost's weight-update mechanism only.
X_synthetic = np.array([[1], [2], [3], [4], [5], [6], [7], [8]])
y_synthetic = np.array([0, 0, 0, 1, 0, 1, 1, 1])  # the point at x=5 is a noisy label (0 among 1s)
n_synthetic = len(y_synthetic)

sample_weights = np.full(n_synthetic, 1.0 / n_synthetic)
print("[SYNTHETIC] Initial sample weights (uniform):", sample_weights.round(4))

stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
stump.fit(X_synthetic, y_synthetic, sample_weight=sample_weights)
stump_pred = stump.predict(X_synthetic)
misclassified = stump_pred != y_synthetic

print("[SYNTHETIC] Stump predictions :", stump_pred)
print("[SYNTHETIC] True labels       :", y_synthetic)
print("[SYNTHETIC] Misclassified mask:", misclassified)


In [ ]:
# [SYNTHETIC] AdaBoost's weight update formula (binary, SAMME-style): continues the toy example above.
weighted_error = sample_weights[misclassified].sum()
alpha = 0.5 * np.log((1 - weighted_error) / weighted_error)

updated_weights = sample_weights * np.exp(alpha * np.where(misclassified, 1, -1))
updated_weights = updated_weights / updated_weights.sum()

print(f"[SYNTHETIC] Weighted error (iteration 1): {weighted_error:.4f}")
print(f"[SYNTHETIC] Estimator weight (alpha)     : {alpha:.4f}")
print("[SYNTHETIC] Updated sample weights (after iteration 1):", updated_weights.round(4))


**[SYNTHETIC] Yorum:** Tek bir gözlem (`x=5`, gürültülü etiket) yanlış sınıflandırıldı. Bu
gözlemin ağırlığı, güncellemeden önce `0.125` (diğerleriyle eşit) idi; güncellemeden sonra `0.50`'ye
yükseldi — yani **toplam ağırlığın yarısı artık bu tek, zor gözleme** ait. Bu, AdaBoost'un "zor
örneklere daha fazla önem verme" mekanizmasının somut bir gösterimidir. Aynı zamanda **Sensitivity
to Noise** kavramını da gösterir: eğer bu nokta gerçek bir gürültü/hata ise (doğru etiket değilse),
AdaBoost ona orantısız bir önem vermiş olur.

### Hands-On — AdaBoost on the Real Dataset


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_cfg = config["lab07"]["models"]["adaboost"]
print("scikit-learn version (API compatibility check):", __import__("sklearn").__version__)
print("AdaBoost configuration:", ada_cfg)

adaboost_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=ada_cfg["n_estimators"],
        learning_rate=ada_cfg["learning_rate"],
        random_state=RANDOM_STATE,
    )),
]).fit(X_train, y_train)

pred_val_ada = adaboost_model.predict(X_val)
proba_val_ada = adaboost_model.predict_proba(X_val)[:, 1]
train_acc_ada = accuracy_score(y_train, adaboost_model.predict(X_train))
print(f"Train Acc={train_acc_ada:.4f}  Val Acc={accuracy_score(y_val, pred_val_ada):.4f}  "
      f"Val F1={f1_score(y_val, pred_val_ada):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val_ada):.4f}")


### Interpretation

AdaBoost (50 decision stump, `learning_rate=0.5`) Val Acc≈0.6321, Val ROC-AUC≈0.6919 ile
Section 3'teki Random Forest'tan (Val Acc≈0.6715, Val ROC-AUC≈0.7406) **daha düşük** performans
gösteriyor. Bu, **"Boosting her zaman Bagging'den daha iyidir" gibi bir iddianın doğru olmadığını**
somut olarak gösteriyor — bu spesifik dataset, feature seti ve hyperparametre konfigürasyonu
altında Random Forest burada daha iyi sonuç veriyor. Bu karşılaştırmayı Section 6'da, aynı
protokolle bütün modelleri bir araya getirerek tekrar sistematik olarak ele alacağız.

### Part C — Gradient Boosting

### Mathematical Explanation

Gradient Boosting, genel bir **additive model** çerçevesi kullanır:

$$F_m(x)=F_{m-1}(x)+\eta\gamma_mh_m(x)$$

- $F_m$: Güncellenmiş ensemble model (m. adımdan sonra)
- $F_{m-1}$: Önceki model
- $h_m$: Yeni weak learner (genellikle küçük bir Decision Tree)
- $\eta$: Learning rate (shrinkage)
- $\gamma_m$: Adıma özgü katkı katsayısı

**Regression için** (squared-error loss), $h_m$ genellikle önceki modelin **residual**'larına
(gerçek değer - tahmin) fit edilir — bu, "her adım öncekinin hatasını tahmin etmeye çalışır"
sezgisinin matematiksel temelidir.

**Classification için**, model doğrudan sınıf residual'larını tahmin etmez; bunun yerine, seçilen
**loss fonksiyonunun (örn. log-loss) negatif gradyanı** üzerinden, her adımda modelin bu gradyanı
en çok azaltacak yönde **aşamalı olarak güncellenir**. "Residual fitting" sezgisi regression'dan
geliyor; classification'da genelleştirilmiş hali **gradient-based (negative gradient) fitting**'tir.

### Hands-On


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_cfg = config["lab07"]["models"]["gradient_boosting"]

gradient_boosting_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=gb_cfg["n_estimators"],
        learning_rate=gb_cfg["learning_rate"],
        max_depth=gb_cfg["max_depth"],
        random_state=RANDOM_STATE,
    )),
]).fit(X_train, y_train)

pred_val_gb = gradient_boosting_model.predict(X_val)
proba_val_gb = gradient_boosting_model.predict_proba(X_val)[:, 1]
train_acc_gb = accuracy_score(y_train, gradient_boosting_model.predict(X_train))
print(f"Train Acc={train_acc_gb:.4f}  Val Acc={accuracy_score(y_val, pred_val_gb):.4f}  "
      f"Val F1={f1_score(y_val, pred_val_gb):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val_gb):.4f}")


### Experiment — Learning Rate


In [ ]:
learning_rate_values = [0.01, 0.1, 0.3]
lr_rows = []

for lr in learning_rate_values:
    start = perf_counter()
    model = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=gb_cfg["n_estimators"],
            learning_rate=lr,
            max_depth=gb_cfg["max_depth"],
            random_state=RANDOM_STATE,
        )),
    ])
    model.fit(X_train, y_train)
    training_time = perf_counter() - start

    pred_tr = model.predict(X_train)
    pred_v = model.predict(X_val)
    proba_v = model.predict_proba(X_val)[:, 1]
    lr_rows.append({
        "learning_rate": lr,
        "Train_Accuracy": accuracy_score(y_train, pred_tr),
        "Validation_Accuracy": accuracy_score(y_val, pred_v),
        "Validation_F1": f1_score(y_val, pred_v),
        "Validation_ROC_AUC": roc_auc_score(y_val, proba_v),
        "Training_Time_s": training_time,
    })

lr_df = pd.DataFrame(lr_rows).round(4)
lr_df


### Interpretation

`n_estimators=100` sabit tutulduğunda, learning rate arttıkça (`0.01→0.3`) **Train Accuracy
artıyor** (0.6627→0.7626, model training verisine giderek daha fazla uyuyor) ama **Validation
ROC-AUC azalıyor** (0.7546→0.7111) — yani yüksek learning rate + sabit estimator sayısı, bu
konfigürasyonda **daha fazla overfitting ve daha zayıf sıralama kalitesi**ne yol açıyor. Dikkat
çekici bir nokta: Validation F1 bu örüntüyü **doğrusal takip etmiyor** (`lr=0.01`'de F1≈0.7248,
`lr=0.1`'de F1≈0.6461, `lr=0.3`'te F1≈0.6643) — çünkü F1, sabit `0.5` threshold'unda hesaplanıyor
ve düşük learning rate'te modelin olasılık çıktıları daha az "keskin" (daha düşük standart sapma),
bu da `0.5` threshold'unda farklı bir precision/recall dengesi üretiyor. **Bu, learning rate ve
estimator sayısının birlikte değerlendirilmesi gerektiğinin somut bir örneğidir** — tek bir
metriğe (örn. sadece F1) bakarak "düşük learning rate daha iyi" sonucuna atlamak, ROC-AUC'nin
gösterdiği genelleme kalitesi düşüşünü gözden kaçırabilir.

### Visualization — Boosting Stage vs Log Loss


In [ ]:
from sklearn.metrics import log_loss

gb_estimator = gradient_boosting_model.named_steps["model"]
X_train_proc = gradient_boosting_model.named_steps["preprocessing"].transform(X_train)
X_val_proc = gradient_boosting_model.named_steps["preprocessing"].transform(X_val)

train_losses = []
val_losses = []
for stage_proba_train, stage_proba_val in zip(
    gb_estimator.staged_predict_proba(X_train_proc), gb_estimator.staged_predict_proba(X_val_proc)
):
    train_losses.append(log_loss(y_train, stage_proba_train[:, 1]))
    val_losses.append(log_loss(y_val, stage_proba_val[:, 1]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(range(1, len(train_losses) + 1), train_losses, color=BAR_COLOR, linewidth=2, label="Training Log Loss")
ax.plot(range(1, len(val_losses) + 1), val_losses, color=WARNING_COLOR, linewidth=2, label="Validation Log Loss")
ax.set_title("Gradient Boosting — Log Loss by Boosting Stage")
ax.set_xlabel("Boosting Stage (number of trees)")
ax.set_ylabel("Log Loss")
ax.legend(frameon=False)
style_axes(ax)
plt.show()

print(f"Training Log Loss   -> stage 1: {train_losses[0]:.4f}  stage {len(train_losses)}: {train_losses[-1]:.4f}")
print(f"Validation Log Loss -> stage 1: {val_losses[0]:.4f}  stage {len(val_losses)}: {val_losses[-1]:.4f}  "
      f"min: {min(val_losses):.4f} (stage {np.argmin(val_losses) + 1})")


### Interpretation

Training Log Loss, boosting aşamaları ilerledikçe **sürekli** azalıyor (model training verisine
giderek daha iyi uyuyor). Validation Log Loss ise bunu bir noktaya kadar takip ediyor, ancak **her
zaman düşmeyebilir** — grafikte minimum validation log loss değerine bir ara aşamada ulaşılabilir,
sonrasında hafifçe artabilir (yukarıdaki çıktıdaki `min` satırına bakın). Bu, Gradient Boosting'in
**çok fazla aşamanın overfitting'e yol açabileceğinin** doğrudan bir göstergesidir ve Early
Stopping'in (Section 5'te kısaca tanıtılacak) neden faydalı olabileceğini motive eder.

### Critical Comparison

AdaBoost ve Gradient Boosting, her ikisi de Boosting ailesindendir ama mekanizmaları farklıdır:
AdaBoost **örnek ağırlıklarını** güncelleyerek zor örneklere odaklanır; Gradient Boosting ise her
adımda **loss fonksiyonunun negatif gradyanına** fit eder. Bu LAB'ın gerçek verisinde Gradient
Boosting (Val ROC-AUC≈0.7338, `lr=0.1`) AdaBoost'tan (Val ROC-AUC≈0.6919) daha iyi performans
gösterdi — ama bu da genel bir kural değil, bu dataset ve konfigürasyonlar için gözlemlenen bir
sonuçtur.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: ada_cfg["learning_rate"] değerini 0.5'ten 1.0'a değiştirip AdaBoostClassifier'ı yeniden
#       eğitin.
# TODO: Val Accuracy/F1/ROC-AUC değerlerinin nasıl değiştiğini yorumlayın.

# ada_lr1 = Pipeline(steps=[("preprocessing", preprocessor), ("model", AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=ada_cfg["n_estimators"], learning_rate=1.0, random_state=RANDOM_STATE))]).fit(X_train, y_train)
# print(accuracy_score(y_val, ada_lr1.predict(X_val)))


---

## Break
**Süre: 10 dakika**

Kısa bir ara verin. Devam ederken Section 5'te XGBoost ve LightGBM'i, bu LAB'ın en güncel ve
üretimde en yaygın kullanılan Gradient Boosting implementasyonlarını inceleyeceğiz.

---


## Section 5 — XGBoost & LightGBM
**Süre: 25 dakika**

### Part A — XGBoost Fundamentals

### Learning Objective

XGBoost'un Gradient Boosting ailesine ait, onu mühendislik açısından (regularization,
hesaplama verimliliği) genişleten bir implementasyon olduğunu anlamak.

### Concept Explanation

**XGBoost (Extreme Gradient Boosting)**, Section 4'teki Gradient Boosting'in **aynı additive
model** mantığını kullanır — bağımsız bir öğrenme paradigması değildir. Ek olarak şunları getirir:

- **First-Order and Second-Order Gradient Information**: Klasik Gradient Boosting yalnızca
  gradyanı (birinci derece türev) kullanırken, XGBoost ayrıca **Hessian**'ı (ikinci derece türev)
  da kullanarak her adımda daha bilgili bir optimizasyon yapar.
- **Regularization**: Objective function'a doğrudan bir **karmaşıklık cezası (penalty)** eklenir
  (aşağıda gösteriliyor) — bu, overfitting'i sınırlamaya yardımcı olur.
- **Tree Pruning Concepts**: XGBoost, ağaçları büyütürken belirli bir kazanç eşiğinin altında
  kalan split'leri "budar" (prune).
- **Histogram-Based Tree Construction** (`tree_method="hist"`): Sürekli feature'ları, split
  noktalarını aramak için **histogram kovalarına** böler — bu, büyük veri kümelerinde ham
  (exact) split aramaya kıyasla çok daha hızlıdır.
- **Parallel Computation / Computational Efficiency**: Histogram oluşturma ve split arama
  adımları paralelleştirilebilir.

### Mathematical Concept

Basitleştirilmiş **Regularized Objective Function**:

$$\mathcal{L}=\sum_{i=1}^{n}l(y_i,\hat{y}_i)+\sum_{k=1}^{K}\Omega(f_k)$$

- $l$: Prediction loss (örn. log-loss)
- $\Omega$: Model complexity penalty (ağaç yaprak sayısı ve yaprak ağırlıkları üzerinden)
- $f_k$: Ensemble içerisindeki k. ağaç
- $K$: Ağaç sayısı

Bu, **kavramsal bir gösterimdir** — gerçek XGBoost objective'i, $\Omega$ içinde yaprak sayısı
($\gamma T$) ve yaprak ağırlıklarının L2 normu ($\frac{1}{2}\lambda\|w\|^2$) gibi terimler
içerir; `reg_lambda` hyperparametresi bu $\lambda$'ya karşılık gelir.

### Hands-On


In [ ]:
from xgboost import XGBClassifier

xgb_cfg = config["lab07"]["models"]["xgboost"]
print("xgboost version:", __import__("xgboost").__version__)
print("XGBoost configuration:", xgb_cfg)

xgboost_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        n_estimators=xgb_cfg["n_estimators"],
        max_depth=xgb_cfg["max_depth"],
        learning_rate=xgb_cfg["learning_rate"],
        subsample=xgb_cfg["subsample"],
        colsample_bytree=xgb_cfg["colsample_bytree"],
        reg_lambda=xgb_cfg["reg_lambda"],
        tree_method="hist",
        device="cpu",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=BENCHMARK_THREADS,
    )),
]).fit(X_train, y_train)

pred_val_xgb = xgboost_model.predict(X_val)
proba_val_xgb = xgboost_model.predict_proba(X_val)[:, 1]
train_acc_xgb = accuracy_score(y_train, xgboost_model.predict(X_train))
print(f"Train Acc={train_acc_xgb:.4f}  Val Acc={accuracy_score(y_val, pred_val_xgb):.4f}  "
      f"Val F1={f1_score(y_val, pred_val_xgb):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val_xgb):.4f}")


### Important Hyperparameters

- **`n_estimators`**: Ağaç (boosting round) sayısı.
- **`max_depth`**: Her ağacın maksimum derinliği.
- **`learning_rate`**: Shrinkage — Section 4'teki $\eta$ ile aynı kavram.
- **`subsample`**: Her ağaç için satırların rastgele alt örneklenme oranı (Bagging'deki bootstrap
  mantığına benzer bir çeşitlendirme kaynağı, ama **replacement olmadan**).
- **`colsample_bytree`**: Her ağaç için feature'ların rastgele alt örneklenme oranı (Random
  Forest'ın `max_features` mantığına benzer).
- **`reg_alpha`** (L1) / **`reg_lambda`** (L2): Yaprak ağırlıkları üzerindeki regularization.
- **`gamma`**: Bir split'in gerçekleşmesi için gereken minimum kayıp azalımı (pruning eşiği).
- **`min_child_weight`**: Bir yaprakta gereken minimum Hessian toplamı (örnek sayısına benzer bir
  rol oynar).
- **`tree_method`**: Ağaç oluşturma algoritması (`"hist"` → histogram tabanlı, büyük veri için
  önerilen).

Bu parametrelerin **tümünü optimize etmeye çalışmıyoruz** — bu LAB'ın amacı, temel
hyperparametrelerin model davranışı üzerindeki etkisini **kavramsal olarak** anlamaktır.

### Part B — LightGBM Fundamentals

### Concept Explanation

**LightGBM (Light Gradient Boosting Machine)**, XGBoost gibi Gradient Boosting ailesindendir,
ancak farklı mühendislik tercihleri yapar:

- **Histogram-Based Learning**: XGBoost'un `tree_method="hist"` seçeneğine benzer şekilde,
  LightGBM **varsayılan olarak** histogram tabanlı split aramayı kullanır.
- **Leaf-Wise Tree Growth**: LightGBM, ağacı **seviye seviye (level-wise)** değil, **en çok kayıp
  azalımı sağlayacak yaprağı** seçip onu büyüterek genişletir. Bu genellikle daha hızlı yakınsama
  sağlar, ama kontrolsüz bırakılırsa (özellikle küçük veri kümelerinde) **daha derin, dengesiz ve
  overfitting'e daha yatkın ağaçlar** üretebilir.
- **Leaf Complexity**: `num_leaves`, bir ağaçtaki toplam yaprak sayısını sınırlar — leaf-wise
  büyümede `max_depth`'ten daha doğrudan bir karmaşıklık kontrolüdür.
- **Large Dataset Efficiency / Training Speed / Memory Efficiency**: LightGBM, özellikle büyük
  veri kümelerinde histogram tabanlı yaklaşımı ve leaf-wise büyümesi sayesinde hızlı eğitim
  sağlayabilir — ama bu LAB'ın küçük veri kümesinde (≈11K satır) bu avantaj **belirgin
  olmayabilir** (Section 6'da gerçek ölçümlerle karşılaştıracağız).
- **Overfitting Risk**: Leaf-wise büyüme, küçük veri kümelerinde `num_leaves`/`min_child_samples`
  dikkatli ayarlanmazsa overfitting riskini artırabilir.

**Not:** XGBoost her durumda yalnızca level-wise büyüme kullanmaz — `tree_method="hist"` ile
XGBoost da yaprak odaklı büyüme seçeneklerine sahiptir (`grow_policy="lossguide"`). Burada
kullandığımız **varsayılan** konfigürasyonlar (XGBoost level-wise, LightGBM leaf-wise) bu iki
kütüphanenin **tipik/varsayılan** davranışını temsil eder, kesin bir mimari sınır değildir.

### Hands-On


In [ ]:
from lightgbm import LGBMClassifier

lgbm_cfg = config["lab07"]["models"]["lightgbm"]
print("lightgbm version:", __import__("lightgbm").__version__)
print("LightGBM configuration:", lgbm_cfg)

lightgbm_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=lgbm_cfg["n_estimators"],
        learning_rate=lgbm_cfg["learning_rate"],
        max_depth=lgbm_cfg["max_depth"],
        num_leaves=lgbm_cfg["num_leaves"],
        min_child_samples=lgbm_cfg["min_child_samples"],
        random_state=RANDOM_STATE,
        n_jobs=BENCHMARK_THREADS,
        verbosity=-1,
    )),
]).fit(X_train, y_train)

pred_val_lgbm = lightgbm_model.predict(X_val)
proba_val_lgbm = lightgbm_model.predict_proba(X_val)[:, 1]
train_acc_lgbm = accuracy_score(y_train, lightgbm_model.predict(X_train))
print(f"Train Acc={train_acc_lgbm:.4f}  Val Acc={accuracy_score(y_val, pred_val_lgbm):.4f}  "
      f"Val F1={f1_score(y_val, pred_val_lgbm):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val_lgbm):.4f}")


### Interpretation

Bu konfigürasyonlarla: XGBoost Val ROC-AUC≈0.7314, LightGBM Val ROC-AUC≈0.7168 — XGBoost burada
hafifçe daha iyi, ancak LightGBM'in Train Accuracy'si (0.7592) XGBoost'tan (0.7102) **daha
yüksek**, bu da LightGBM'in bu konfigürasyonla training verisine biraz daha fazla uyduğunu
(muhtemelen leaf-wise büyüme + `num_leaves=31`'in bu küçük veri kümesi için nispeten yüksek
olması nedeniyle) gösteriyor. **Bu, "LightGBM her dataset'te en hızlı/en iyi modeldir" gibi genel
bir iddianın burada doğrulanmadığının** bir örneğidir — Section 6'da bütün modelleri (hız dahil)
sistematik olarak karşılaştıracağız.

### Important Hyperparameters

- **`num_leaves`**: Bir ağaçtaki maksimum yaprak sayısı — leaf-wise büyümede birincil karmaşıklık
  kontrolü.
- **`max_depth`**: Üst sınır olarak da kullanılabilir (leaf-wise büyümeyle birlikte).
- **`n_estimators` / `learning_rate`**: Section 4 ile aynı kavramlar.
- **`min_child_samples`**: Bir yaprakta gereken minimum örnek sayısı (overfitting'e karşı bir
  frenleme mekanizması).
- **`max_bin`**: Histogram kovası sayısı (hassasiyet/hız dengesi).
- **`feature_fraction`** / **`bagging_fraction`**: XGBoost'un `colsample_bytree`/`subsample`'ına
  karşılık gelen alt örnekleme oranları.

**`num_leaves` ve `max_depth` ilişkisi:** Tam, dengeli bir ağaçta `num_leaves ≈ 2^max_depth`
olur. Ancak leaf-wise büyümede ağaçlar **dengesiz** olabilir — yani `num_leaves=31` ile
`max_depth=5` birlikte kullanıldığında, ağaç teorik `2^5=32` sınırına yakın yaprak sayısına
derinlik sınırından **önce** ulaşabilir, bu da `max_depth`'in pratikte her zaman baskın kısıt
olmayabileceği anlamına gelir. **Küçük veri kümelerinde kontrolsüz leaf-wise büyüme** (çok yüksek
`num_leaves`), modelin training verisindeki gürültüyü ezberlemesine yol açabilir — bu yüzden
`min_child_samples` gibi frenleme parametreleri önemlidir.

### Early Stopping

**Early Stopping**, boosting sürecini sabit bir `n_estimators` değerine kadar değil, validation
performansı belirli bir sayıda ardışık round boyunca iyileşmediğinde **otomatik olarak**
durduran bir tekniktir. Bu LAB'ın ana benchmark'ında (Section 6) bütün boosting modelleri **önceden
belirlenmiş, sabit `n_estimators` değerleriyle** eğitilecektir — Early Stopping zorunlu
uygulanmayacaktır. Aşağıda, **isteğe bağlı bir örnek** olarak XGBoost için Early Stopping
gösterilmektedir; bu, Section 6'nın ana benchmark tablosundaki sabit-estimator XGBoost
sonucundan **ayrı** raporlanmaktadır.

**Doğru yöntem:** `eval_set`, test setini **değil**, validation setini kullanmalı; ayrıca
validation verisi, estimator'a gönderilmeden önce **sadece training verisine fit edilmiş**
preprocessor ile dönüştürülmelidir (Scikit-learn `Pipeline`'ın `eval_set` parametresini otomatik
olarak preprocess ettiği varsayılmamalıdır — aşağıda bunu manuel olarak yapıyoruz).


In [ ]:
# Optional Early Stopping example (XGBoost) — kept separate from the main fixed-n_estimators
# benchmark in Section 6. We fit the preprocessor on X_train ONLY, then reuse it (transform-only)
# for X_val before passing it as eval_set — Pipeline.fit's eval_set is NOT auto-preprocessed.
X_train_for_es = preprocessor.fit_transform(X_train, y_train)
X_val_for_es = preprocessor.transform(X_val)

xgb_early_stopping_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=300,
    max_depth=xgb_cfg["max_depth"],
    learning_rate=xgb_cfg["learning_rate"],
    subsample=xgb_cfg["subsample"],
    colsample_bytree=xgb_cfg["colsample_bytree"],
    reg_lambda=xgb_cfg["reg_lambda"],
    tree_method="hist",
    device="cpu",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=BENCHMARK_THREADS,
    early_stopping_rounds=20,
)
xgb_early_stopping_model.fit(X_train_for_es, y_train, eval_set=[(X_val_for_es, y_val)], verbose=False)

pred_val_es = xgb_early_stopping_model.predict(X_val_for_es)
proba_val_es = xgb_early_stopping_model.predict_proba(X_val_for_es)[:, 1]
print(f"Early-stopped at boosting round: {xgb_early_stopping_model.best_iteration} (cap was 300, patience=20)")
print(f"[Early Stopping, NOT part of Section 6 benchmark] Val Acc={accuracy_score(y_val, pred_val_es):.4f}  "
      f"Val F1={f1_score(y_val, pred_val_es):.4f}  Val ROC-AUC={roc_auc_score(y_val, proba_val_es):.4f}")


Early Stopping burada 52. round'da durdu (sabırsızlık penceresi 20 round) ve Val ROC-AUC'yi
(≈0.7390) sabit `n_estimators=100` XGBoost modeline (≈0.7314) kıyasla hafifçe iyileştirdi. **Ancak
bu sonuç, validation verisinin hem durma noktası kararında hem de performans ölçümünde
kullanılması nedeniyle, sabit-estimator modellerle birebir aynı koşullarda elde edilmedi** — bu
yüzden Section 6'nın ana karşılaştırma tablosuna dahil edilmeyecek, Challenge 3'te ayrıca
genişletilecektir.

### Critical Rules

- XGBoost'un her zaman en başarılı model olduğu iddia edilemez.
- LightGBM'in her dataset üzerinde en hızlı model olduğu iddia edilemez.
- Hesaplama performansı veri boyutuna, donanıma ve model konfigürasyonuna bağlıdır.
- Aynı `n_estimators` değeri, farklı kütüphaneler/algoritmalar arasında eşit model kapasitesi veya
  eşit hesaplama bütçesi anlamına gelmez.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: lgbm_cfg["num_leaves"] değerini 31'den 7'ye düşürüp LGBMClassifier'ı yeniden eğitin.
# TODO: Train Accuracy / Val Accuracy / Generalization Gap'in nasıl değiştiğini yorumlayın.

# lgbm_small_leaves = Pipeline(steps=[("preprocessing", preprocessor), ("model", LGBMClassifier(objective="binary", n_estimators=lgbm_cfg["n_estimators"], learning_rate=lgbm_cfg["learning_rate"], max_depth=lgbm_cfg["max_depth"], num_leaves=7, min_child_samples=lgbm_cfg["min_child_samples"], random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS, verbosity=-1))]).fit(X_train, y_train)
# print(accuracy_score(y_train, lgbm_small_leaves.predict(X_train)), accuracy_score(y_val, lgbm_small_leaves.predict(X_val)))


---


## Section 6 — Comparative Model Benchmarking
**Süre: 25 dakika**

**Bu LAB'ın en önemli bölümü.**

### Benchmark Objective

Section 2-5'te tanıttığımız modelleri, **aynı deney koşullarında** karşılaştırmak.

### Required Models

1. `DummyClassifier`
2. `LogisticRegression`
3. `DecisionTreeClassifier`
4. `RandomForestClassifier`
5. `AdaBoostClassifier`
6. `GradientBoostingClassifier`
7. `XGBClassifier`
8. `LGBMClassifier`

(`BaggingClassifier`, Section 2 Part D'de yalnızca demonstrasyon amaçlıydı — ana benchmark'a dahil
edilmiyor, çünkü Section 2'de zaten tek ağaca kıyasla etkisini gösterdik.)

### Benchmark Protocol

Bütün modeller için **aynı**: dataset, feature set, training split, validation split, target
tanımı, positive class, evaluation metrikleri, decision threshold (`0.5`), CPU thread politikası
(`BENCHMARK_THREADS`), belgelenmiş hyperparametreler (`config/settings.yaml`) ve reproducible
random seed (`RANDOM_STATE=42`) kullanılacaktır.

**Önemli:** Model konfigürasyonları (Section 2-5'te belirlenen `config/settings.yaml`
değerleri) **sonuçlar görülmeden önce** belirlendi — bu, kontrollü bir deneydir, kapsamlı bir
hyperparameter optimizasyonu değildir.


In [ ]:
from sklearn.metrics import confusion_matrix, average_precision_score

print("Additional metrics imported for Section 6:", "confusion_matrix, average_precision_score")


In [ ]:
def build_pipeline(model_name):
    # Returns a FRESH, UNFITTED Pipeline(preprocessing + model) for the given model name.
    if model_name == "DummyClassifier":
        estimator = DummyClassifier(strategy="most_frequent")
    elif model_name == "LogisticRegression":
        estimator = LogisticRegression(
            max_iter=config["lab07"]["models"]["logistic_regression"]["max_iter"], random_state=RANDOM_STATE
        )
    elif model_name == "DecisionTreeClassifier":
        estimator = DecisionTreeClassifier(
            max_depth=config["lab07"]["models"]["decision_tree"]["max_depth"], random_state=RANDOM_STATE
        )
    elif model_name == "RandomForestClassifier":
        estimator = RandomForestClassifier(
            n_estimators=rf_cfg["n_estimators"], max_depth=rf_cfg["max_depth"],
            min_samples_leaf=rf_cfg["min_samples_leaf"], max_features=rf_cfg["max_features"],
            random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
        )
    elif model_name == "AdaBoostClassifier":
        estimator = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1), n_estimators=ada_cfg["n_estimators"],
            learning_rate=ada_cfg["learning_rate"], random_state=RANDOM_STATE,
        )
    elif model_name == "GradientBoostingClassifier":
        estimator = GradientBoostingClassifier(
            n_estimators=gb_cfg["n_estimators"], learning_rate=gb_cfg["learning_rate"],
            max_depth=gb_cfg["max_depth"], random_state=RANDOM_STATE,
        )
    elif model_name == "XGBClassifier":
        estimator = XGBClassifier(
            objective="binary:logistic", n_estimators=xgb_cfg["n_estimators"], max_depth=xgb_cfg["max_depth"],
            learning_rate=xgb_cfg["learning_rate"], subsample=xgb_cfg["subsample"],
            colsample_bytree=xgb_cfg["colsample_bytree"], reg_lambda=xgb_cfg["reg_lambda"],
            tree_method="hist", device="cpu", eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
        )
    elif model_name == "LGBMClassifier":
        estimator = LGBMClassifier(
            objective="binary", boosting_type="gbdt", n_estimators=lgbm_cfg["n_estimators"],
            learning_rate=lgbm_cfg["learning_rate"], max_depth=lgbm_cfg["max_depth"],
            num_leaves=lgbm_cfg["num_leaves"], min_child_samples=lgbm_cfg["min_child_samples"],
            random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS, verbosity=-1,
        )
    else:
        raise ValueError(f"Unknown model_name: {model_name}")
    return Pipeline(steps=[("preprocessing", preprocessor), ("model", estimator)])


MODEL_FAMILY = {
    "DummyClassifier": "Baseline",
    "LogisticRegression": "Linear",
    "DecisionTreeClassifier": "Tree",
    "RandomForestClassifier": "Bagging",
    "AdaBoostClassifier": "Boosting",
    "GradientBoostingClassifier": "Boosting",
    "XGBClassifier": "Boosting",
    "LGBMClassifier": "Boosting",
}

BENCHMARK_MODEL_NAMES = list(MODEL_FAMILY.keys())
print("Models to benchmark:", BENCHMARK_MODEL_NAMES)


### Part A — Evaluation Metrics & Decision Threshold Policy

**Decision Threshold Policy (ana benchmark için ortak):** `predict_proba()[:, positive_class] >=
0.5` ise sınıf `1` tahmin edilir (`>=`, yani olasılık **tam olarak** `0.5` olduğunda sınıf `1`
seçilir — bu tie-breaking kuralı bütün modellerde tutarlı uygulanır). LAB06'da optimize edilen
threshold (`≈0.53`), yalnızca o LAB'daki tek bir model (Logistic Regression) için seçilmişti; bunu
burada farklı model ailelerine **adil olmayan** bir biçimde tekrar uygulamıyoruz. Threshold
Optimization ve Probability Calibration bu LAB'ın konusu değildir.

`predict_proba()`'nın hangi sütununun pozitif sınıfa karşılık geldiğini **asla varsaymıyoruz** —
her model için `classes_` dizisinden doğruluyoruz.


In [ ]:
def extract_positive_class_scores(pipeline, X, positive_class):
    # Returns predict_proba scores for the configured positive class, verified via classes_.
    classes = list(pipeline.named_steps["model"].classes_)
    class_index = classes.index(positive_class)
    return pipeline.predict_proba(X)[:, class_index]


def evaluate_classifier(y_true, pred, proba):
    # Computes the Section 6 validation metrics for one model's predictions.
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    _, fp, fn, _ = cm.ravel()
    return {
        "Validation_Precision": precision_score(y_true, pred, zero_division=0),
        "Validation_Recall": recall_score(y_true, pred, zero_division=0),
        "Validation_F1": f1_score(y_true, pred, zero_division=0),
        "Validation_ROC_AUC": roc_auc_score(y_true, proba),
        "Validation_AP": average_precision_score(y_true, proba),
        "False_Positive_Count": int(fp),
        "False_Negative_Count": int(fn),
    }


### Part B — Bias–Variance & Generalization

`Generalization_Gap = Train_Accuracy - Validation_Accuracy` olarak tanımlanır. **Bu, Bias–Variance
Decomposition'ın istatistiksel/matematiksel bir ölçümü değildir** — gözlemsel bir model performans
karşılaştırmasıdır. Büyük bir gap şu nedenlerden kaynaklanabilir: Model Complexity (çok karmaşık
bir model), Overfitting, Noise (veri gürültüsü), veya Data Distribution Differences (train/val
dağılım farkları — bu LAB'da stratified split sayesinde minimize edilmiştir).

### Part C — Training Time Measurement

`time.perf_counter()` ile, **end-to-end Pipeline Training Time** (preprocessing fit + estimator
fit, ikisi birlikte) ölçülür — yalnızca estimator değil. Her ölçüm **3 kez tekrarlanır** ve
**median** değer raporlanır; her tekrarda `build_pipeline()` ile **yeni, unfit edilmiş** bir
pipeline oluşturulur (aynı nesne üzerinde warm-start/tekrar eğitim **yapılmaz**).


In [ ]:
def measure_training_time(model_name, X_tr, y_tr, repeats=None):
    # Median end-to-end Pipeline training time over `repeats` fresh, unfitted clones.
    repeats = repeats or config["lab07"]["benchmark"]["training_repeats"]
    times = []
    fitted_pipeline = None
    for _ in range(repeats):
        pipe = build_pipeline(model_name)
        start = perf_counter()
        pipe.fit(X_tr, y_tr)
        times.append(perf_counter() - start)
        fitted_pipeline = pipe  # keep the last fit for evaluation/inference measurement
    return float(np.median(times)), fitted_pipeline


### Part D — Inference Latency Measurement

Aynı validation batch'i (`X_val`, tüm 1650 satır) üzerinde her modelin `predict_proba()` süresi
ölçülür. **Ölçümden önce bir warm-up prediction** yapılır (JIT/cache etkilerini dışarıda tutmak
için), ardından ölçüm **birden fazla kez tekrarlanır** ve **median** değer kullanılır. Metrik
hesaplama ve plotting işlemleri timer'ın **dışında** tutulur. `Inference_Time_Per_Sample_ms`,
batch süresinden türetilmiştir — **gerçek bir single-request API latency'si değildir**.


In [ ]:
def measure_inference_latency(fitted_pipeline, X, repeats=None, warmup=None):
    # Median predict_proba() batch latency (seconds), after warm-up runs.
    repeats = repeats or config["lab07"]["benchmark"]["inference_repeats"]
    warmup = warmup if warmup is not None else config["lab07"]["benchmark"]["warmup_runs"]
    for _ in range(warmup):
        fitted_pipeline.predict_proba(X)
    batch_times = []
    for _ in range(repeats):
        start = perf_counter()
        fitted_pipeline.predict_proba(X)
        batch_times.append(perf_counter() - start)
    return float(np.median(batch_times))


### Benchmark Environment (recorded for reproducibility)


In [ ]:
import platform

print("scikit-learn:", __import__("sklearn").__version__)
print("xgboost     :", __import__("xgboost").__version__)
print("lightgbm    :", __import__("lightgbm").__version__)
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("Python      :", platform.python_version())
print("os.cpu_count():", __import__("os").cpu_count())
print("Fixed n_jobs / thread policy used for RandomForest/XGBoost/LightGBM:", BENCHMARK_THREADS)
print("Docker resource limits: none explicitly configured in compose.yaml (host/VM defaults apply)")


### Part E — Benchmark Results DataFrame


In [ ]:
benchmark_rows = []
for model_name in BENCHMARK_MODEL_NAMES:
    training_time_s, fitted_pipeline = measure_training_time(model_name, X_train, y_train)

    pred_train = fitted_pipeline.predict(X_train)
    proba_val = extract_positive_class_scores(fitted_pipeline, X_val, POSITIVE_CLASS)
    pred_val = (proba_val >= DECISION_THRESHOLD).astype(int)
    val_metrics = evaluate_classifier(y_val, pred_val, proba_val)

    batch_latency_s = measure_inference_latency(fitted_pipeline, X_val)

    row = {
        "Model": model_name,
        "Model_Family": MODEL_FAMILY[model_name],
        "Train_Accuracy": accuracy_score(y_train, pred_train),
        "Validation_Accuracy": accuracy_score(y_val, pred_val),
        **val_metrics,
        "Training_Time_s": training_time_s,
        "Batch_Inference_Latency_ms": batch_latency_s * 1000,
        "Inference_Time_Per_Sample_ms": (batch_latency_s * 1000) / len(X_val),
        "Throughput_samples_per_s": len(X_val) / batch_latency_s,
    }
    row["Generalization_Gap"] = row["Train_Accuracy"] - row["Validation_Accuracy"]
    benchmark_rows.append(row)
    print("Benchmarked:", model_name)

benchmark_df = pd.DataFrame(benchmark_rows)[[
    "Model", "Model_Family", "Train_Accuracy", "Validation_Accuracy", "Validation_Precision",
    "Validation_Recall", "Validation_F1", "Validation_ROC_AUC", "Validation_AP",
    "False_Positive_Count", "False_Negative_Count", "Generalization_Gap", "Training_Time_s",
    "Batch_Inference_Latency_ms", "Inference_Time_Per_Sample_ms", "Throughput_samples_per_s",
]]
benchmark_df.round(4)


### Interpretation

**Discrimination ranking (Validation ROC-AUC, bu LAB'ın önceden belirlenmiş ranking metric'i):**
`RandomForestClassifier` (≈0.741) en yüksek, onu `GradientBoostingClassifier` (≈0.734) ve
`XGBClassifier` (≈0.731) takip ediyor; `DecisionTreeClassifier` (≈0.729) ve `LogisticRegression`
(≈0.725) şaşırtıcı biçimde yakın; `LGBMClassifier` (≈0.717) ve özellikle `AdaBoostClassifier`
(≈0.692) bu dataset+konfigürasyon için en zayıf ayırt etme gücüne sahip.

**Kritik uyarı — F1'e güvenmeyin:** `DummyClassifier`'ın Validation F1'i (≈0.748) listedeki **en
yüksek** değer — Section 7, Threshold Selection'daki aynı tuzak burada da geçerli: positive class
(`1`) çoğunlukta olduğu için, her zaman `1` diyen bir model yüksek Recall (1.0) ve prevalence'a
yakın bir Precision (≈0.597) ile yapay olarak yüksek bir F1 üretir. **Bu, tek bir metriğe (özellikle
F1) bakarak model seçmenin riskini doğrudan gösterir** — ROC-AUC'nin (`DummyClassifier`'da tam
olarak 0.500, yani sıfır ayırt etme gücü) burada çok daha güvenilir bir sinyal olduğu açıktır. Bu
yüzden ranking metric'i (ROC-AUC) **sonuçları görmeden önce** tanımlamıştık.

**Generalization Gap:** En yüksek gap `LGBMClassifier`'da (≈0.100) — Section 5'teki gözlemimizle
tutarlı biçimde, bu konfigürasyonla training verisine en fazla uyan model. `RandomForestClassifier`
(≈0.070) ve `GradientBoostingClassifier` (≈0.043) orta düzeyde; `LogisticRegression` hafifçe
**negatif** bir gap'e sahip (≈-0.005, training'den biraz daha iyi validation — küçük örnekleme
gürültüsü, gerçek bir "negatif overfitting" değil).

**Training Time / Inference Latency — bir ölçüm gürültüsü notu:** Yukarıdaki tablodaki Accuracy/
F1/ROC-AUC sütunları (sabit `random_state` sayesinde) her çalıştırmada **tam olarak** aynı
değerleri üretir; ancak `Training_Time_s` ve `Inference_*_latency` sütunları, işletim sistemi
zamanlaması, arka plan süreçleri ve CPU frequency scaling nedeniyle **aynı makinede farklı
çalıştırmalar arasında bile değişebilir**. Bu yüzden kesin rakamlar yerine, çalıştırmalar arasında
**tutarlı biçimde gözlemlenen** büyüklük mertebesi ve sıralama örüntülerine odaklanıyoruz:

- **Training Time**: `GradientBoostingClassifier`, sıralı (paralelleştirilemeyen) doğası nedeniyle
  **her çalıştırmada en yavaş eğiten model** oluyor — diğer bütün modellerden belirgin biçimde
  (tipik olarak birkaç katından fazla) daha yavaş. `DummyClassifier`, `LogisticRegression` ve
  `DecisionTreeClassifier` tutarlı biçimde **en hızlı** eğitilen modeller. `XGBClassifier` ve
  `LGBMClassifier`, histogram tabanlı inşa sayesinde `GradientBoostingClassifier`'dan **tutarlı
  biçimde daha hızlı** — bu, Section 5'teki "histogram tabanlı inşa daha verimlidir" iddiasını bu
  ölçek ve donanımda somut olarak doğruluyor.
- **Inference Latency**: `RandomForestClassifier` (100 ağacın her birinin tahmin üretmesi
  gerektiği için) ve `AdaBoostClassifier` (50 sıralı stump), tutarlı biçimde **en yüksek**
  per-sample latency'ye sahip modeller arasında; `DummyClassifier`, `LogisticRegression` ve
  `DecisionTreeClassifier` tutarlı biçimde **en düşük** latency'ye sahip.

Yukarıdaki hücrenin **gerçek çıktısındaki** kesin sayılar, bu çalıştırmaya özgüdür ve farklı bir
çalıştırmada (aynı makinede bile) birkaç kat değişebilir — bu, benchmark raporlarken mutlak
sayıların değil, **göreli sıralamanın ve büyüklük mertebesinin** vurgulanması gerektiğinin
somut bir örneğidir.

**Önemli hatırlatma:** Bu ölçümler, bu spesifik donanım (12 CPU, Docker container, `n_jobs=2`
politikası) ve bu küçük dataset (≈11K satır) için geçerlidir — **farklı donanım veya veri
boyutlarında evrensel bir performans sonucu olarak sunulmamalıdır.**

### Part F — Benchmark Visualizations


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sorted_f1 = benchmark_df.sort_values("Validation_F1", ascending=True)
ax.barh(sorted_f1["Model"], sorted_f1["Validation_F1"], color=BAR_COLOR)
ax.set_title("Validation F1-Score by Model")
ax.set_xlabel("Validation F1")
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sorted_auc = benchmark_df.sort_values("Validation_ROC_AUC", ascending=True)
colors = [HIGHLIGHT_COLOR if m == "RandomForestClassifier" else BAR_COLOR for m in sorted_auc["Model"]]
ax.barh(sorted_auc["Model"], sorted_auc["Validation_ROC_AUC"], color=colors)
ax.axvline(0.5, color=WARNING_COLOR, linewidth=1.5, linestyle="--", label="Random guessing (0.5)")
ax.set_title("Validation ROC-AUC by Model (ranking metric)")
ax.set_xlabel("Validation ROC-AUC")
ax.legend(frameon=False)
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(benchmark_df))
width = 0.35
ax.bar(x_pos - width / 2, benchmark_df["Train_Accuracy"], width, color=BAR_COLOR, label="Train Accuracy")
ax.bar(x_pos + width / 2, benchmark_df["Validation_Accuracy"], width, color=WARNING_COLOR, label="Validation Accuracy")
ax.set_xticks(x_pos)
ax.set_xticklabels(benchmark_df["Model"], rotation=40, ha="right")
ax.set_title("Training vs Validation Accuracy")
ax.set_ylabel("Accuracy")
ax.legend(frameon=False)
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sorted_time = benchmark_df.sort_values("Training_Time_s", ascending=True)
ax.barh(sorted_time["Model"], sorted_time["Training_Time_s"], color=BAR_COLOR)
ax.set_title("Training Time by Model (median of 3 runs)")
ax.set_xlabel("Training Time (seconds, log scale)")
ax.set_xscale("log")  # log scale: training times span two orders of magnitude (ms to ~1s)
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sorted_latency = benchmark_df.sort_values("Inference_Time_Per_Sample_ms", ascending=True)
ax.barh(sorted_latency["Model"], sorted_latency["Inference_Time_Per_Sample_ms"], color=BAR_COLOR)
ax.set_title("Inference Latency by Model (per-sample, median of 10 runs)")
ax.set_xlabel("Inference Time per Sample (ms, log scale)")
ax.set_xscale("log")  # log scale: latencies span roughly two orders of magnitude
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for _, row in benchmark_df.iterrows():
    color = HIGHLIGHT_COLOR if row["Model"] == "RandomForestClassifier" else BAR_COLOR
    ax.scatter(row["Training_Time_s"], row["Validation_ROC_AUC"], color=color, s=80, zorder=3)
    ax.annotate(row["Model"], (row["Training_Time_s"], row["Validation_ROC_AUC"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Training Time (seconds, log scale)")
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Predictive Performance vs Training Time (Performance-Cost Trade-off)")
style_axes(ax)
plt.tight_layout()
plt.show()


Son grafik, bu LAB'ın temel mesajını özetliyor: **en yüksek Validation ROC-AUC'ye sahip model
(`RandomForestClassifier`), en hızlı eğitilen model değil** (ama en yavaş olan
`GradientBoostingClassifier`'dan da hızlı) — yani burada "performans için orantısız bir maliyet
ödemiyoruz" diyebiliriz. Ancak `LogisticRegression` ve `DecisionTreeClassifier`, çok daha düşük bir
eğitim maliyetiyle **ROC-AUC açısından neredeyse aynı bölgede** duruyor — bu, Section 7'nin
"ek karmaşıklık her zaman haklı mı?" sorusunu doğrudan motive ediyor.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: benchmark_df'i Validation_AP (Average Precision) sütununa göre sıralayın.
# TODO: Bu sıralamanın Validation_ROC_AUC sıralamasıyla ne kadar örtüştüğünü karşılaştırın.

# print(benchmark_df[["Model", "Validation_ROC_AUC", "Validation_AP"]].sort_values("Validation_AP", ascending=False))


---


## Section 7 — Model Selection & Final Workshop
**Süre: 15 dakika**

### Workshop Objective

Section 6'daki benchmark sonuçlarından hareketle, teknik ve operasyonel bir model seçimi yapmak.

### Model Selection Criteria

- Predictive Performance (Validation ROC-AUC — **önceden tanımlanmış ana ranking metric**)
- Generalization Performance (Generalization Gap)
- Precision–Recall Trade-off (iş hedefine bağlı)
- Training Cost
- Inference Latency
- Model Complexity
- Explainability
- Maintainability
- Business Requirements

Average Precision, F1, Precision ve Recall **tamamlayıcı** metrikler olarak değerlendirilir —
Section 6'da gördüğümüz gibi, F1 tek başına (özellikle `DummyClassifier` örneğinde) yanıltıcı
olabilir. Gerçek iş maliyetleri bu LAB kapsamında bilinmediği için **belirli parasal değerler
uydurulmamaktadır**.

### Student Questions

Aşağıdaki sorular, Section 6'nın `benchmark_df`'inden **gerçek verilerle** cevaplanabilir.


In [ ]:
print("1) En yüksek Validation ROC-AUC:")
print(benchmark_df.loc[benchmark_df["Validation_ROC_AUC"].idxmax(), ["Model", "Validation_ROC_AUC"]])
print()
print("2) En yüksek Validation F1:")
print(benchmark_df.loc[benchmark_df["Validation_F1"].idxmax(), ["Model", "Validation_F1"]])
print()
print("3) En hızlı eğitilen model:")
print(benchmark_df.loc[benchmark_df["Training_Time_s"].idxmin(), ["Model", "Training_Time_s"]])
print()
print("4) En düşük inference latency (per-sample):")
print(benchmark_df.loc[benchmark_df["Inference_Time_Per_Sample_ms"].idxmin(), ["Model", "Inference_Time_Per_Sample_ms"]])
print()
print("5) En güçlü overfitting belirtisi (en yüksek Generalization Gap):")
print(benchmark_df.loc[benchmark_df["Generalization_Gap"].idxmax(), ["Model", "Generalization_Gap"]])


In [ ]:
logreg_auc = benchmark_df.loc[benchmark_df["Model"] == "LogisticRegression", "Validation_ROC_AUC"].iloc[0]
best_auc_row = benchmark_df.loc[benchmark_df["Validation_ROC_AUC"].idxmax()]

print("6) Daha karmaşık bir model, LogisticRegression'a göre anlamlı bir iyileşme sağladı mı?")
print(f"   LogisticRegression Val ROC-AUC = {logreg_auc:.4f}")
print(f"   En iyi model ({best_auc_row['Model']}) Val ROC-AUC = {best_auc_row['Validation_ROC_AUC']:.4f}")
print(f"   Fark: {best_auc_row['Validation_ROC_AUC'] - logreg_auc:+.4f} ROC-AUC puanı")
print()
print("7) Random Forest ile Gradient Boosting arasındaki temel fark:")
print("   Random Forest: bağımsız (bootstrap + random feature) ağaçların PARALEL ortalaması (Bagging).")
print("   Gradient Boosting: ardışık ağaçların, önceki hatanın negatif gradyanına göre SIRALI eklenmesi (Boosting).")
print()
xgb_auc = benchmark_df.loc[benchmark_df["Model"] == "XGBClassifier", "Validation_ROC_AUC"].iloc[0]
lgbm_auc = benchmark_df.loc[benchmark_df["Model"] == "LGBMClassifier", "Validation_ROC_AUC"].iloc[0]
gb_auc = benchmark_df.loc[benchmark_df["Model"] == "GradientBoostingClassifier", "Validation_ROC_AUC"].iloc[0]
print("8) XGBoost ve LightGBM, ek karmaşıklığı haklı çıkaracak değer üretti mi?")
print(f"   XGBoost Val ROC-AUC={xgb_auc:.4f}  LightGBM Val ROC-AUC={lgbm_auc:.4f}  "
      f"(karşılaştırma: klasik GradientBoosting={gb_auc:.4f}, LogisticRegression={logreg_auc:.4f})")
print("   Bu dataset ve konfigürasyonda XGBoost/LightGBM, klasik GradientBoosting'i geçemedi;")
print("   yani burada ek mühendislik karmaşıklığı ölçülebilir bir ROC-AUC avantajıyla sonuçlanmadı.")


**9) Production ortamında hangi modeli tercih ederdiniz? Neden?** — Bu sorunun tek bir doğru
cevabı yoktur; cevap, aşağıdaki **Model Selection Cases**'teki iş önceliğine bağlıdır.

### Model Selection Cases


In [ ]:
print("Case A — Real-Time Classification (düşük inference latency önceliklidir):")
case_a_naive = benchmark_df.loc[benchmark_df["Inference_Time_Per_Sample_ms"].idxmin()]
print(f"  Naif seçim (sadece latency'ye göre): {case_a_naive['Model']}  "
      f"(Latency={case_a_naive['Inference_Time_Per_Sample_ms']:.4f} ms/sample, "
      f"Val ROC-AUC={case_a_naive['Validation_ROC_AUC']:.4f})")

# A real deployment candidate needs a latency floor on discrimination, not just low latency --
# otherwise a trivial, non-discriminating model (ROC-AUC = 0.5) looks like the "best" choice.
real_models = benchmark_df[benchmark_df["Model"] != "DummyClassifier"]
case_a = real_models.loc[real_models["Inference_Time_Per_Sample_ms"].idxmin()]
print(f"  Gerçekçi seçim (DummyClassifier hariç, gerçek ayırt etme gücüne sahip modeller arasında):")
print(f"  Seçilen: {case_a['Model']}  (Latency={case_a['Inference_Time_Per_Sample_ms']:.4f} ms/sample, "
      f"Val ROC-AUC={case_a['Validation_ROC_AUC']:.4f})")

print()
print("Case B — Predictive Performance (tahmin performansı önceliklidir, daha yüksek eğitim")
print("maliyeti kabul edilebilir):")
case_b = benchmark_df.loc[benchmark_df["Validation_ROC_AUC"].idxmax()]
print(f"  Seçilen: {case_b['Model']}  (Val ROC-AUC={case_b['Validation_ROC_AUC']:.4f}, "
      f"Training_Time_s={case_b['Training_Time_s']:.4f})")

print()
print("Case C — Maintainability & Explainability (anlaşılabilirlik, bakım kolaylığı, operasyonel")
print("sadelik önceliklidir):")
case_c_candidates = benchmark_df[benchmark_df["Model"].isin(["LogisticRegression", "DecisionTreeClassifier"])]
case_c = case_c_candidates.loc[case_c_candidates["Validation_ROC_AUC"].idxmax()]
print(f"  Seçilen: {case_c['Model']}  (katsayılar/split kuralları doğrudan yorumlanabilir; "
      f"Val ROC-AUC={case_c['Validation_ROC_AUC']:.4f})")


### Interpretation

**Case A'daki naif sonuç önemli bir uyarı içeriyor:** Sadece inference latency'ye göre seçim
yapıldığında `DummyClassifier` "kazanıyor" — ama bu model sıfır ayırt etme gücüne sahip
(ROC-AUC=0.5000). Bu, Section 6'daki F1/`DummyClassifier` tuzağının **birebir aynısı**: tek bir
metriği (burada latency), bir **minimum performans eşiği** olmadan optimize etmek, anlamsız/dejenere
bir sonuca yakınsayabilir. Gerçekçi Case A seçimi, önce "gerçek bir ayırt etme gücüne sahip mi?"
sorusunu sorup, **sonra** bu adaylar arasında en düşük latency'yi arar.

Üç senaryo, (dejenere seçimler elendikten sonra) **üç farklı modeli** haklı çıkarabiliyor: Case A
ve Case B genellikle farklı modelleri öne çıkarır (en düşük latency'ye sahip **gerçekçi** model,
en yüksek ROC-AUC'ye sahip model ile aynı olmak zorunda değildir — Section 6'nın
Performance-Cost Trade-off grafiğine bakın); Case C'de ise açıkça `LogisticRegression` veya
`DecisionTreeClassifier` gibi, katsayıları/split kurallarını doğrudan okuyabileceğiniz bir model
tercih edilir. **Bu, "en iyi model" diye tek bir evrensel cevabın olmadığının somut bir
gösterimidir** — doğru cevap, iş önceliğine (ve her zaman bir minimum performans eşiğine) bağlıdır.

### Evaluation Integrity

Section 2'de belirttiğimiz gibi, bu LAB'ın `X_test`/`y_test`'i LAB06'da **zaten bir kez**
kullanılmış, gerçekten bağımsız bir holdout değildir. Bu yüzden:

- Section 6-7'deki **bütün model seçim kararları, yalnızca validation verisi üzerinden**
  verilmiştir.
- Bu LAB'da, seçilen model adayı üzerinde **ek bir "nihai test evaluation" yapılmamaktadır** —
  çünkü gerçekten görülmemiş, seçim kararlarına dahil edilmemiş ayrı bir holdout bulunmuyor.
- Bu nedenle Section 6'nın benchmark tablosu, bir **development/validation benchmark'ı** olarak
  raporlanmaktadır; bağımsız bir "production-ready final evaluation" olarak sunulmamaktadır.

### Expected Final Report — Benchmark Findings


---

## Benchmark Findings

### Dataset & Business Problem

**Dataset:** E-Commerce Shipping Data (`data/ecommerce_shipping/Train.csv`, 10999 sipariş, LAB02-
LAB06 ile aynı). **İş Problemi:** Shipment Delay Classification — `Reached.on.Time_Y.N` (`1` = Late
Delivery, positive class; `0` = On Time).

### Experiment Configuration

- Feature set: `Cost_of_the_Product, Prior_purchases, Discount_offered, Weight_in_gms`
  (numerical), `Warehouse_block, Mode_of_Shipment, Product_importance` (categorical) — LAB06 ile
  birebir aynı.
- Split: Stratified, `random_state=42`, `%70/%15/%15` (train/val/test) — **LAB06'dakiyle index
  seviyesinde aynı satırlar** (Section 2'de doğrulandı).
- Decision Threshold: `0.5` (ortak, bütün modeller için).
- CPU thread politikası: `n_jobs=2` (RandomForest/XGBoost/LightGBM için); kontrolsüz `n_jobs=-1`
  kullanılmadı.
- Ranking metric: Validation ROC-AUC (sonuçlar görülmeden önce tanımlandı).

### Baseline Performance

`DummyClassifier(most_frequent)`: Val Accuracy≈0.597, Val ROC-AUC=0.500 (sıfır ayırt etme gücü),
ama Val F1≈0.748 (**en yüksek F1** — pozitif sınıfın çoğunlukta olması nedeniyle yanıltıcı; bkz.
Section 6 Interpretation).

### Compared Models

8 model: `DummyClassifier`, `LogisticRegression`, `DecisionTreeClassifier`,
`RandomForestClassifier`, `AdaBoostClassifier`, `GradientBoostingClassifier`, `XGBClassifier`,
`LGBMClassifier`.

### Predictive Performance

Validation ROC-AUC sıralaması: `RandomForestClassifier` ilk sırada, onu `GradientBoostingClassifier`
ve `XGBClassifier` yakından takip ediyor; `DecisionTreeClassifier` ve `LogisticRegression` bu
üçlüye şaşırtıcı derecede yakın; `LGBMClassifier` ve `AdaBoostClassifier` bu dataset+konfigürasyon
için en zayıf ayırt etme gücüne sahip (tam değerler için Section 6'nın `benchmark_df`'ine bakın —
sabit `random_state` sayesinde bu sıralama her çalıştırmada aynıdır).

### Generalization Analysis

`LGBMClassifier`, bu konfigürasyonla en yüksek Generalization Gap'e sahip — leaf-wise büyüme +
küçük veri kümesi kombinasyonunun beklenen etkisiyle tutarlı. `LogisticRegression` en düşük
(hafifçe negatif) gap'e sahip. Bu gap değerleri, **istatistiksel bir Bias-Variance Decomposition
değil**, gözlemsel bir karşılaştırmadır.

### Training Cost

`GradientBoostingClassifier`, sıralı (paralelleştirilemeyen) yapısı nedeniyle **tutarlı biçimde**
en yavaş eğitilen model. `XGBClassifier`/`LGBMClassifier`, histogram tabanlı inşa sayesinde ondan
belirgin biçimde daha hızlı. Kesin saniye değerleri çalıştırmalar arasında değişebilir (Section 6'da
not edildi) — bu yüzden burada **göreli sıralama** vurgulanmaktadır.

### Inference Performance

`RandomForestClassifier` (100 ağaç) ve `AdaBoostClassifier` (50 sıralı stump), tutarlı biçimde en
yüksek per-sample latency'ye sahip; `DummyClassifier`/`LogisticRegression`/`DecisionTreeClassifier`
en düşük latency'ye sahip. Bu ölçümler, tek bir batch (1650 satır) üzerinden türetilmiş ortalama
değerlerdir — gerçek bir single-request API latency'si değildir.

### Selected Model Candidate

Bu LAB, **tek bir evrensel "seçilen model"** önermez — Model Selection Cases (Case A/B/C) üç farklı
operasyonel önceliğe göre üç farklı adayı haklı çıkarabiliyor. Ranking metric'e (Validation
ROC-AUC) göre genel bir aday aranıyorsa, bu LAB'ın çalıştırmasında `RandomForestClassifier` en
yüksek puanı almıştır.

### Selection Rationale

Seçim, **yalnızca validation verisi** üzerinden, önceden tanımlanmış bir ranking metric (ROC-AUC)
ve tamamlayıcı metriklerle (F1, AP, Generalization Gap, Training Time, Inference Latency) yapıldı.
F1 tek başına kullanılsaydı `DummyClassifier` yanlışlıkla "en iyi model" olarak görünürdü — bu,
neden birden fazla metriğin birlikte değerlendirilmesi gerektiğinin somut bir kanıtıdır.

### Dataset Limitations

- `Customer_rating`/`Customer_care_calls` (leakage riski) ve `Gender` (fairness riski) model
  girdisi olarak kullanılmadı (LAB06 ile tutarlı).
- Dataset tek bir zaman dilimine/coğrafyaya ait olabilir; genellenebilirliği doğrulanmamıştır.

### Benchmark Limitations

- `X_test`/`y_test`, LAB06'da zaten kullanılmış olduğu için bu LAB'da **bağımsız bir final test
  evaluation yapılmamıştır** — bütün sonuçlar development/validation benchmark'ı olarak
  raporlanmaktadır.
- Hyperparametreler **kapsamlı olarak optimize edilmemiştir** — bu, kontrollü bir başlangıç
  konfigürasyonu karşılaştırmasıdır, bir "en iyi model" iddiası değildir.
- Training Time/Inference Latency ölçümleri bu spesifik donanım ve veri boyutu için geçerlidir;
  farklı donanım/ölçekte farklı sonuçlar beklenebilir.

### Future Improvements

- LAB09'da ele alınacak kapsamlı Hyperparameter Optimization (GridSearchCV/RandomizedSearchCV/
  Optuna bu LAB'da **kasıtlı olarak** uygulanmadı).
- LAB08'de ele alınacak Feature Engineering ve Explainability (SHAP bu LAB'da **kasıtlı olarak**
  kullanılmadı).
- Gerçekten bağımsız, hiç görülmemiş bir test kümesi elde edilirse, seçilen model adayının o
  küme üzerinde tek seferlik bir final evaluation'ı yapılabilir.

---


## Advanced Challenges (İsteğe Bağlı)

Aşağıdaki dört challenge, ana **150 dakikalık** programın **dışındadır** ve isteğe bağlıdır.
Deneyimli katılımcılar için tasarlanmıştır.

### Challenge 1 — Ensemble Size & Diminishing Returns

Random Forest için `n_estimators`'ı Section 3'tekinden daha geniş bir aralıkta kontrollü biçimde
artırıp, performans/maliyet ilişkisini inceleyelim.


In [ ]:
challenge1_rows = []
for n in [10, 25, 50, 100, 200, 400]:
    start = perf_counter()
    model = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=n, max_depth=rf_cfg["max_depth"], min_samples_leaf=rf_cfg["min_samples_leaf"],
            max_features=rf_cfg["max_features"], random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
        )),
    ])
    model.fit(X_train, y_train)
    training_time = perf_counter() - start
    proba_v = model.predict_proba(X_val)[:, 1]
    challenge1_rows.append({"n_estimators": n, "Validation_ROC_AUC": roc_auc_score(y_val, proba_v), "Training_Time_s": training_time})

challenge1_df = pd.DataFrame(challenge1_rows)
challenge1_df["Marginal_AUC_per_100_extra_trees"] = (
    challenge1_df["Validation_ROC_AUC"].diff() / (challenge1_df["n_estimators"].diff() / 100)
)
challenge1_df.round(5)


**Yorum:** Validation ROC-AUC, `n_estimators=10`'da zaten en yüksek seviyeye yakın; `25`'ten
`400`'e kadar **anlamlı bir iyileşme göstermiyor**, hatta bazı noktalarda hafifçe düşüyor
(`Marginal_AUC_per_100_extra_trees` sütununa bakın — çoğu değer negatif veya sıfıra yakın).
Training Time ise `n_estimators` ile **neredeyse doğrusal** artıyor. Bu, klasik "diminishing
returns" (azalan getiri) eğrisinden biraz farklı, daha çarpıcı bir bulgu: bu dataset için ek
ağaçların **hiçbir zaman** anlamlı bir katkı sağlamadığı görülüyor — getiri, ilk birkaç ağaçtan
sonra zaten düzleşmiş durumda. **Bu, bu spesifik dataset/feature seti için geçerlidir; genel bir
kural olarak sunulmamaktadır.**

### Challenge 2 — Equal Computational Budget

Random Forest, XGBoost ve LightGBM'i **yaklaşık eşit** bir training time bütçesi altında
karşılaştıralım. Önce referans bütçeyi (Section 6'daki Random Forest konfigürasyonunun eğitim
süresi) ölçüyoruz, ardından XGBoost ve LightGBM için, bu bütçeye **en yakın** eğitim süresini
veren `n_estimators` değerini küçük bir arama ile buluyoruz.

**Dikkat:** Training Time ölçümü gürültülüdür (Section 6'da not edildi) — bu yüzden **kesin eşit**
bir hesaplama maliyeti elde edildiği iddia edilmemektedir; bu, yalnızca **yaklaşık** bir eşleştirmedir.


In [ ]:
reference_start = perf_counter()
reference_rf = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=rf_cfg["n_estimators"], max_depth=rf_cfg["max_depth"], min_samples_leaf=rf_cfg["min_samples_leaf"],
        max_features=rf_cfg["max_features"], random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
    )),
]).fit(X_train, y_train)
reference_budget_s = perf_counter() - reference_start
reference_auc = roc_auc_score(y_val, reference_rf.predict_proba(X_val)[:, 1])
print(f"Reference budget (RandomForestClassifier, n_estimators={rf_cfg['n_estimators']}): "
      f"{reference_budget_s:.4f}s, Val ROC-AUC={reference_auc:.4f}")


def find_closest_budget_match(build_fn, candidate_n_estimators, target_budget_s):
    best = None
    for n in candidate_n_estimators:
        start = perf_counter()
        model = build_fn(n).fit(X_train, y_train)
        elapsed = perf_counter() - start
        diff = abs(elapsed - target_budget_s)
        if best is None or diff < best["diff"]:
            best = {"n_estimators": n, "Training_Time_s": elapsed, "diff": diff, "model": model}
    return best


In [ ]:
xgb_match = find_closest_budget_match(
    lambda n: Pipeline(steps=[("preprocessing", preprocessor), ("model", XGBClassifier(
        objective="binary:logistic", n_estimators=n, max_depth=xgb_cfg["max_depth"],
        learning_rate=xgb_cfg["learning_rate"], subsample=xgb_cfg["subsample"],
        colsample_bytree=xgb_cfg["colsample_bytree"], reg_lambda=xgb_cfg["reg_lambda"],
        tree_method="hist", device="cpu", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
    ))]),
    candidate_n_estimators=[50, 100, 150, 200, 300, 400],
    target_budget_s=reference_budget_s,
)

lgbm_match = find_closest_budget_match(
    lambda n: Pipeline(steps=[("preprocessing", preprocessor), ("model", LGBMClassifier(
        objective="binary", boosting_type="gbdt", n_estimators=n, learning_rate=lgbm_cfg["learning_rate"],
        max_depth=lgbm_cfg["max_depth"], num_leaves=lgbm_cfg["num_leaves"], min_child_samples=lgbm_cfg["min_child_samples"],
        random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS, verbosity=-1,
    ))]),
    candidate_n_estimators=[200, 400, 600, 800, 1000, 1200],
    target_budget_s=reference_budget_s,
)

budget_comparison = pd.DataFrame([
    {"Model": "RandomForestClassifier", "n_estimators": rf_cfg["n_estimators"],
     "Training_Time_s": reference_budget_s, "Validation_ROC_AUC": reference_auc},
    {"Model": "XGBClassifier", "n_estimators": xgb_match["n_estimators"],
     "Training_Time_s": xgb_match["Training_Time_s"],
     "Validation_ROC_AUC": roc_auc_score(y_val, xgb_match["model"].predict_proba(X_val)[:, 1])},
    {"Model": "LGBMClassifier", "n_estimators": lgbm_match["n_estimators"],
     "Training_Time_s": lgbm_match["Training_Time_s"],
     "Validation_ROC_AUC": roc_auc_score(y_val, lgbm_match["model"].predict_proba(X_val)[:, 1])},
])
budget_comparison.round(4)


**Yorum:** Yukarıdaki tablo, **yaklaşık** eşit bir eğitim süresi bütçesi altında üç modeli
karşılaştırıyor (`Training_Time_s` sütunundaki değerler birbirine yakın ama **özdeş değil** —
arama, aday listesindeki en yakın eşleşmeyi buluyor, kesin bir eşitlik garantisi yok). Bu
karşılaştırma, Section 6'nın **varsayılan `n_estimators=100`** karşılaştırmasından farklı bir
soruya cevap veriyor: "Eğer üç modele de yaklaşık aynı hesaplama bütçesini verirsek, hangisi daha
iyi performans gösterir?" — bu, "aynı `n_estimators` = aynı hesaplama bütçesi" **değildir**
prensibinin (master prompt §5 Part E) doğrudan bir uygulamasıdır.

### Challenge 3 — Early Stopping (Expanded)

Section 5'te XGBoost için bir Early Stopping örneği göstermiştik. Şimdi LightGBM için de aynı
yaklaşımı (train-only fit edilmiş preprocessor, validation verisi `eval_X`/`eval_y` olarak) uygulayıp,
**sabit estimator sayılı** ve **early-stopped** modelleri ayrı bir tabloda karşılaştıralım.


In [ ]:
from lightgbm import early_stopping

lgbm_early_stopping_model = LGBMClassifier(
    objective="binary", boosting_type="gbdt", n_estimators=300,
    learning_rate=lgbm_cfg["learning_rate"], max_depth=lgbm_cfg["max_depth"],
    num_leaves=lgbm_cfg["num_leaves"], min_child_samples=lgbm_cfg["min_child_samples"],
    random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS, verbosity=-1,
)
# Reuses X_train_for_es / X_val_for_es from Section 5 (preprocessor fit on train only).
lgbm_early_stopping_model.fit(
    X_train_for_es, y_train, eval_X=X_val_for_es, eval_y=y_val,
    callbacks=[early_stopping(stopping_rounds=20, verbose=False)],
)

print(f"LightGBM early-stopped at boosting round: {lgbm_early_stopping_model.best_iteration_} (cap was 300, patience=20)")


In [ ]:
proba_val_lgbm_es = lgbm_early_stopping_model.predict_proba(X_val_for_es)[:, 1]
pred_val_lgbm_es = (proba_val_lgbm_es >= DECISION_THRESHOLD).astype(int)

early_stopping_comparison = pd.DataFrame([
    {"Model": "XGBClassifier (fixed n_estimators=100)",
     "Validation_ROC_AUC": benchmark_df.loc[benchmark_df["Model"] == "XGBClassifier", "Validation_ROC_AUC"].iloc[0]},
    {"Model": "XGBClassifier (early-stopped, cap=300)",
     "Validation_ROC_AUC": roc_auc_score(y_val, xgb_early_stopping_model.predict_proba(X_val_for_es)[:, 1])},
    {"Model": "LGBMClassifier (fixed n_estimators=100)",
     "Validation_ROC_AUC": benchmark_df.loc[benchmark_df["Model"] == "LGBMClassifier", "Validation_ROC_AUC"].iloc[0]},
    {"Model": "LGBMClassifier (early-stopped, cap=300)",
     "Validation_ROC_AUC": roc_auc_score(y_val, proba_val_lgbm_es)},
]).round(4)
early_stopping_comparison


**Yorum:** Her iki kütüphanede de Early Stopping, sabit `n_estimators=100` modeline kıyasla
Validation ROC-AUC'yi **hafifçe** iyileştiriyor. **Ancak bu karşılaştırma, Section 6'nın ana
benchmark protokolüyle birebir aynı koşullarda yapılmamıştır** — early-stopped modeller, durma
noktası kararı için **validation verisini görmüştür**, sabit-estimator modeller ise görmemiştir.
Bu yüzden bu tablo, Section 6'nın ana benchmark'ına **eklenmemiş**, ayrı raporlanmıştır (master
prompt §5 Early Stopping kuralı).

### Challenge 4 — Voting Classifier

Section 2-6'da eğittiğimiz üç farklı aileden modeli (`LogisticRegression`, `RandomForestClassifier`,
`XGBClassifier`) birleştiren küçük bir Soft Voting ensemble oluşturalım.


In [ ]:
from sklearn.ensemble import VotingClassifier

voting_soft = VotingClassifier(
    estimators=[
        ("logreg", logreg_model),
        ("random_forest", random_forest_model),
        ("xgboost", xgboost_model),
    ],
    voting="soft",
).fit(X_train, y_train)

proba_val_voting = voting_soft.predict_proba(X_val)[:, 1]
pred_val_voting = (proba_val_voting >= DECISION_THRESHOLD).astype(int)

voting_comparison = pd.DataFrame([
    {"Model": "LogisticRegression", "Validation_ROC_AUC": benchmark_df.loc[benchmark_df["Model"] == "LogisticRegression", "Validation_ROC_AUC"].iloc[0]},
    {"Model": "RandomForestClassifier", "Validation_ROC_AUC": benchmark_df.loc[benchmark_df["Model"] == "RandomForestClassifier", "Validation_ROC_AUC"].iloc[0]},
    {"Model": "XGBClassifier", "Validation_ROC_AUC": benchmark_df.loc[benchmark_df["Model"] == "XGBClassifier", "Validation_ROC_AUC"].iloc[0]},
    {"Model": "Soft VotingClassifier (all 3)", "Validation_ROC_AUC": roc_auc_score(y_val, proba_val_voting)},
]).round(4)
voting_comparison


**Soft Voting Probability Calibration sınırlaması:** Soft Voting, her base model'in
`predict_proba()` çıktısının **basit ortalamasını** alır. Ancak `LogisticRegression`,
`RandomForestClassifier` ve `XGBClassifier`'ın olasılık çıktıları **farklı şekilde kalibre
edilmiştir** (örn. ağaç tabanlı modellerin olasılıkları genellikle `0`/`1`'e daha "sıkışık"
dağılabilir) — hiçbiri açıkça kalibre edilmemiştir (`CalibratedClassifierCV` kullanılmamıştır).
Bu, basit ortalamanın, farklı modellerin güvenilirliğini **örtük olarak eşit** varsaydığı
anlamına gelir; bu her zaman doğru bir varsayım değildir.


In [ ]:
print(f"Soft Voting Val ROC-AUC = {roc_auc_score(y_val, proba_val_voting):.4f}")
print(f"Best individual model (RandomForestClassifier) Val ROC-AUC = "
      f"{benchmark_df.loc[benchmark_df['Model'] == 'RandomForestClassifier', 'Validation_ROC_AUC'].iloc[0]:.4f}")


**Yorum:** Bu çalıştırmada Soft Voting, **üç base model'in her birinden de daha yüksek** bir
Validation ROC-AUC elde etti — bu, model diversity'nin (Section 1) somut bir faydasını gösteriyor:
üç farklı model ailesi (linear, bagging, boosting), muhtemelen kısmen farklı hatalar yapıyor ve
ortalama bu hataları kısmen birbirine karşı dengeliyor. **Ancak bu sonuç genellenmemelidir** —
Voting'in her zaman tekil modellerden daha iyi performans göstereceği **varsayılamaz**; bu,
base model'lerin çeşitliliğine ve bu spesifik dataset'e bağlıdır.

### Student Exercise


In [ ]:
# Advanced Challenge Student Exercise:
# TODO: voting_soft'a dördüncü bir model (örn. decision_tree_model benzeri bir Pipeline) ekleyip
#       Validation ROC-AUC'nin nasıl değiştiğini gözlemleyin.
# TODO: "Daha fazla model eklemek her zaman Voting performansını iyileştirir mi?" sorusunu, bu
#       dataset üzerinde gerçek sayılarla cevaplayın.

# voting_four = VotingClassifier(estimators=[("logreg", logreg_model), ("random_forest", random_forest_model), ("xgboost", xgboost_model), ("tree", tree_model)], voting="soft").fit(X_train, y_train)
# print(roc_auc_score(y_val, voting_four.predict_proba(X_val)[:, 1]))


---

## LAB07 Özeti

Bu LAB'da, LAB02-LAB06'da öğrendiğiniz bilgileri Ensemble Learning ve karşılaştırmalı model
benchmarking üzerinde kullandınız:

- Ensemble Learning'in Bagging ve Boosting ailelerini kavramsal ve matematiksel olarak öğrendiniz.
- Bootstrap Sampling mekanizmasını hem küçük bir örnekle hem Bagging demonstrasyonuyla gördünüz.
- Random Forest'ı inşa edip `n_estimators`/`max_depth`'in gerçek etkisini ölçtünüz (ve "daha fazla
  her zaman daha iyi değildir" prensibini gerçek verilerle doğruladınız).
- AdaBoost'un sample-weight mekanizmasını sentetik bir örnekle, Gradient Boosting'in additive
  model mantığını matematiksel olarak öğrendiniz.
- XGBoost ve LightGBM'i, aralarındaki mühendislik farklarını anlayarak kullandınız.
- Sekiz modeli **aynı protokolle** (metrikler, generalization gap, training time, inference
  latency) karşılaştırdınız ve F1'in (DummyClassifier örneğinde) nasıl yanıltıcı olabileceğini
  somut olarak gördünüz.
- Model seçiminin iş önceliğine bağlı olduğunu üç farklı senaryoyla gösterdiniz.
- İsteğe bağlı challenge'larda ensemble size, equal computational budget, early stopping ve
  voting ensemble'ları inceledeniz.

Bir sonraki oturumda görüşmek üzere!
